# 🇪🇬 Egypt Tech Jobs Aggregator - Enhanced Edition

This notebook fetches tech jobs from **15+ job sources** to maximize opportunities for Egyptian tech professionals.

## 📊 Job Sources

### 🏢 ATS Platforms (Company Career Pages)
- **Greenhouse** - 50+ company boards
- **Lever** - 15+ company sites (case-sensitive!)
- **Workable** - 35+ companies

### 🇪🇬 Egyptian Job Boards
| Source | Type | Description |
|--------|------|-------------|
| **Wuzzuf** | Scraper | Egypt's largest job board |
| **Jooble** | API | Job aggregator with Egypt focus |
| **Forasna** | Scraper | Egyptian job portal |
| **Indeed Egypt** | Scraper | Indeed's Egypt section |
| **Bayt.com** | Scraper | Major MENA job board |
| **LinkedIn** | Scraper | Public guest job search |
| **Jobzella** | Scraper | Egyptian startup jobs |
| **Tanqeeb** | Scraper | Arab region aggregator |

### 🌍 Remote Job Boards (Work from Egypt)
- **RemoteOK** - Global remote tech jobs (API)
- **Remotive** - Curated remote positions (API)
- **Himalayas** - Remote tech roles (API)
- **Jobicy** - Remote-first companies (API)

## 📋 Output Columns
| Column | Description |
|--------|-------------|
| **Title** | Job title |
| **Company** | Company name |
| **Level** | Junior/Mid/Senior/Lead/Principal |
| **Skills** | Detected technical skills |
| **Source** | Job board |
| **Country** | Location country |
| **City** | Egypt city/governorate |
| **Work_Type** | Remote/Hybrid/On-site |
| **Link** | Application URL |
| **Date** | Posted date |

## ⚙️ Configuration
Enable/disable any source in Cell 3. Each scraper can be individually toggled.

In [4]:
# ╔════════════════════════════════════════════════════════════════════════════════════════╗
# ║  EGYPT TECH JOBS AGGREGATOR — Refined Single-Output Version                           ║
# ║  Outputs: DataFrame with Title, Company, Source, Country, Link                        ║
# ╚════════════════════════════════════════════════════════════════════════════════════════╝

import re, sys, subprocess, json
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any, Set, Tuple
from importlib.util import find_spec
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

# ── Ensure requests is available ──
if find_spec("requests") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
import requests

# ── Ensure pandas is available ──
if find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])
import pandas as pd

# ── Ensure BeautifulSoup is available (for Wuzzuf scraping) ──
if find_spec("bs4") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "beautifulsoup4"])
from bs4 import BeautifulSoup

print("✅ Dependencies loaded successfully!")

✅ Dependencies loaded successfully!


In [5]:
import os
# ════════════════════════════════════════════════════════════════════════════════════════
#                              CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════════════

# Filters
EGYPT_ONLY = True           # Only show Egypt-based jobs
TECH_ONLY = True            # Only show tech roles
INCLUDE_REMOTE_EGYPT = True # Include "Remote" jobs if location mentions Egypt
INCLUDE_PRODUCT = True      # Include Product Manager / BA roles
INCLUDE_DESIGN = False      # Include UI/UX roles

# Performance
TIMEOUT = 20                # HTTP timeout in seconds
MAX_WORKERS = 30            # Concurrent threads

# Jooble API Configuration
JOOBLE_API_KEY = os.environ.get("JOOBLE_API_KEY", "")
JOOBLE_ENABLED = True       # Enable Jooble API for additional Egypt jobs
JOOBLE_DAYS_BACK = 14       # Fetch jobs from last N days
JOOBLE_MAX_PAGES = 5        # Max pages per keyword
JOOBLE_SEARCH_KEYWORDS = [
    # ═══════════════════════════════════════════════════════════════════════════
    # DATABASE & SQL DEVELOPER ROLES (Priority Focus)
    # ═══════════════════════════════════════════════════════════════════════════
    
    # Oracle Specific
    "oracle developer", "oracle sql developer", "oracle database developer",
    "oracle pl/sql developer", "plsql developer", "pl/sql developer",
    "oracle programmer", "oracle engineer", "oracle consultant",
    "oracle apex developer", "apex developer", "oracle forms developer",
    "oracle reports developer", "oracle ebs developer", "oracle e-business",
    "oracle fusion developer", "oracle cloud developer", "oracle oci",
    "oracle integration", "oracle soa developer", "oracle middleware",
    "oracle data integrator", "odi developer", "oracle golden gate",
    "oracle adf developer", "adf developer", "oracle weblogic",
    "oracle database", "oracle 19c", "oracle 21c", "oracle 12c",
    
    # SQL Developer Roles (Non-DBA)
    "sql developer", "sql programmer", "sql engineer", "sql analyst",
    "database developer", "database programmer", "database engineer",
    "database analyst", "db developer", "db engineer",
    "t-sql developer", "tsql developer", "transact-sql developer",
    "mysql developer", "postgresql developer", "postgres developer",
    "sql server developer", "mssql developer", "microsoft sql developer",
    "mariadb developer", "sqlite developer",
    
    # Data & ETL Development
    "etl developer", "data integration developer", "data pipeline engineer",
    "ssis developer", "ssrs developer", "ssas developer",
    "informatica developer", "talend developer", "datastage developer",
    "pentaho developer", "data warehouse developer", "dwh developer",
    "data modeling", "database design", "database architect",
    
    # BI & Reporting Development
    "bi developer", "business intelligence developer", "report developer",
    "power bi developer", "tableau developer", "looker developer",
    "cognos developer", "microstrategy developer", "qlik developer",
    "crystal reports developer", "reporting analyst",
    
    # Data Engineering
    "data engineer", "big data developer", "data platform engineer",
    "spark developer", "hadoop developer", "databricks developer",
    "snowflake developer", "redshift developer", "bigquery developer",
    "airflow developer", "kafka developer", "data architect",
    
    # ═══════════════════════════════════════════════════════════════════════════
    # GENERAL SOFTWARE ENGINEERING
    # ═══════════════════════════════════════════════════════════════════════════
    
    # Core Software Engineering
    "software engineer", "software developer", "programmer", "coder",
    "backend developer", "backend engineer", "frontend developer", "frontend engineer",
    "fullstack developer", "full stack engineer", "web developer",
    
    # Programming Languages
    "python developer", "java developer", "javascript developer", "typescript developer",
    "golang developer", "go developer", "rust developer", "c++ developer",
    "php developer", "ruby developer", "scala developer", "kotlin developer",
    ".net developer", "c# developer", "node.js developer", "nodejs developer",
    
    # Frontend & Mobile
    "react developer", "angular developer", "vue developer", "react native developer",
    "mobile developer", "ios developer", "android developer", "flutter developer",
    "swift developer", "ui developer",
    
    # Backend & Infrastructure  
    "devops engineer", "site reliability engineer", "sre", "platform engineer",
    "cloud engineer", "aws engineer", "azure engineer", "gcp engineer",
    "infrastructure engineer", "systems engineer", "linux engineer",
    "kubernetes engineer", "docker", "terraform",
    
    # Data & AI
    "data scientist", "data analyst", "machine learning engineer",
    "ml engineer", "ai engineer", "deep learning", "nlp engineer",
    "analytics engineer",
    
    # Security
    "security engineer", "cybersecurity", "infosec", "penetration tester",
    "soc analyst", "application security",
    
    # QA & Testing
    "qa engineer", "test engineer", "automation engineer", "sdet",
    "quality assurance", "test automation",
    
    # Other Tech Roles
    "solutions architect", "technical lead", "tech lead", "engineering manager",
    "scrum master", "agile coach", "product manager", "technical product manager",
    "business analyst", "systems analyst",
    
    # Remote-specific searches
    "remote software", "remote developer", "remote engineer",
    "remote sql developer", "remote database developer", "remote oracle",
]

# Wuzzuf Configuration (Egypt's largest job board)
WUZZUF_ENABLED = True       # Enable Wuzzuf scraping
WUZZUF_MAX_PAGES = 15       # Max pages to fetch (15 jobs per page)
WUZZUF_DAYS_BACK = 14       # Only include jobs from last N days

# RemoteOK Configuration (Global remote job board - Public API)
REMOTEOK_ENABLED = True     # Enable RemoteOK API
REMOTEOK_TAGS = ["dev", "engineer", "backend", "frontend", "devops", "data", "mobile",
                 "sql", "database", "oracle", "postgres", "mysql", "etl", "bi",
                 "python", "java", "javascript", "react", "angular", "node"]

# Remotive Configuration (Remote tech jobs - Public API)
REMOTIVE_ENABLED = True     # Enable Remotive API
REMOTIVE_CATEGORIES = ["software-dev", "data", "devops-sysadmin", "product", "qa"]

# ══════════════════════════════════════════════════════════════════════════════════════════
#                 ADDITIONAL EGYPTIAN JOB SOURCES (Scrapers)
# ══════════════════════════════════════════════════════════════════════════════════════════

# Forasna Configuration (Egyptian job board)
FORASNA_ENABLED = False
FORASNA_MAX_PAGES = 5
FORASNA_DAYS_BACK = 14

# Tanqeeb Configuration (Arab region job aggregator)
TANQEEB_ENABLED = False  # Currently broken - HTML structure changed

# ══════════════════════════════════════════════════════════════════════════════════════════
# ⛔ PERMANENTLY DISABLED SOURCES - DO NOT ENABLE!
# ══════════════════════════════════════════════════════════════════════════════════════════
#
# The following sources are PERMANENTLY DISABLED for legal and ethical (halal) reasons:
#
# 1. LINKEDIN - Section 8.2.2 & 8.2.4 of LinkedIn User Agreement EXPLICITLY PROHIBITS:
#    - Scraping or copying data from LinkedIn
#    - Using bots or automated methods to access the service
#    - Using data for commercial purposes without permission
#    ➤ Violation can result in legal action and permanent bans
#
# 2. INDEED - Indeed's Terms of Service EXPLICITLY PROHIBITS:
#    - Scraping, crawling, or automated data collection
#    - Reproducing job listings without express permission
#    ➤ Indeed actively pursues legal action against scrapers
#
# 3. BAYT.COM - Terms require review but likely prohibit scraping
#    - Using without clear permission is not halal (ethical)
#    - Better to be safe than risk violating their terms
#
# ══════════════════════════════════════════════════════════════════════════════════════════
# ⛔ PERMANENTLY DISABLED SOURCES CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════════════════

# Indeed Egypt Configuration - ⛔ PERMANENTLY DISABLED (Terms prohibit scraping)
INDEED_EGYPT_ENABLED = False  # DO NOT ENABLE - Violates Terms of Service
INDEED_MAX_PAGES = 5
INDEED_DAYS_BACK = 14

# Bayt.com Configuration - ⛔ PERMANENTLY DISABLED (Terms unclear, not halal to use)
BAYT_ENABLED = False  # DO NOT ENABLE - Terms unclear, not halal to scrape
BAYT_MAX_PAGES = 5
BAYT_DAYS_BACK = 14

# LinkedIn Jobs Configuration - ⛔ PERMANENTLY DISABLED (Explicitly prohibited)
LINKEDIN_ENABLED = False  # DO NOT ENABLE - LinkedIn Terms Section 8.2 prohibits scraping
LINKEDIN_KEYWORDS = ["software engineer", "developer", "data engineer", "devops", "backend", "frontend"]
LINKEDIN_MAX_PAGES = 3

# Jobzella Configuration - ⛔ PERMANENTLY DISABLED (Terms unclear, not halal to use)
JOBZELLA_ENABLED = False  # DO NOT ENABLE - Terms unclear, not halal to scrape

# ══════════════════════════════════════════════════════════════════════════════════════════
# 🕌 HALAL REMINDER:
# In Islam, we must respect agreements and contracts (العقود). Using data from 
# websites that explicitly prohibit it in their Terms of Service violates:
# - The principle of honoring agreements (الوفاء بالعهد)
# - The prohibition against taking what doesn't belong to us
# - The command to deal justly with others
#
# ALTERNATIVE: Use only sources with PUBLIC APIs or explicit permission:
# ✅ Greenhouse, Lever, Workable (public job board APIs)
# ✅ Jooble (partner API with API key)
# ✅ RemoteOK, Remotive, Himalayas, Jobicy (public APIs)
# ✅ Wuzzuf (with respectful rate limiting and attribution)
# ══════════════════════════════════════════════════════════════════════════════════════════

print(f"   Remote Sources: RemoteOK, Remotive, Himalayas, Jobicy")
print("")
print("⛔ PERMANENTLY DISABLED (Legal/Halal Compliance):")
print("   - LinkedIn (Terms Section 8.2 prohibits scraping)")
print("   - Indeed (Terms explicitly prohibit scraping)")
print("   - Bayt.com (Terms unclear - not halal to use)")
print("   - Jobzella (Terms unclear - not halal to use)")

   Remote Sources: RemoteOK, Remotive, Himalayas, Jobicy

⛔ PERMANENTLY DISABLED (Legal/Halal Compliance):
   - LinkedIn (Terms Section 8.2 prohibits scraping)
   - Indeed (Terms explicitly prohibit scraping)
   - Bayt.com (Terms unclear - not halal to use)
   - Jobzella (Terms unclear - not halal to use)


In [6]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                    JOB SOURCES POLICY CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════════════
# This configuration defines the data source policies for legal compliance and attribution.
# 
# ENUMS:
# - source_type: official_api, rss_feed, scraping, manual
# - allowed_mode: full_display, limited_display, link_only, disabled
#
# ENFORCEMENT RULES:
# - full_display: API can return full job details including description
# - limited_display: Return metadata + snippet only (max 200 chars), always include apply_url
# - link_only: Return ONLY minimal fields (title/company/location/date) + apply_url
# - disabled: Exclude from all public results
# ════════════════════════════════════════════════════════════════════════════════════════

from dataclasses import dataclass, field, asdict
from typing import Optional, Literal
from enum import Enum
import json

# ═══════════════════════════════════════════════════════════════════════════
# ENUMS FOR TYPE SAFETY
# ═══════════════════════════════════════════════════════════════════════════

class SourceType(str, Enum):
    OFFICIAL_API = "official_api"
    RSS_FEED = "rss_feed"
    SCRAPING = "scraping"
    MANUAL = "manual"

class AllowedMode(str, Enum):
    FULL_DISPLAY = "full_display"      # Can return full job details
    LIMITED_DISPLAY = "limited_display" # Metadata + 200 char snippet only
    LINK_ONLY = "link_only"            # Only card fields + apply URL
    DISABLED = "disabled"              # Exclude from public results

# ═══════════════════════════════════════════════════════════════════════════
# JOB SOURCE DATACLASS (Schema Definition)
# ═══════════════════════════════════════════════════════════════════════════

@dataclass
class RateLimit:
    """Rate limiting configuration per source."""
    requests_per_minute: int = 60
    burst: int = 10  # Max concurrent requests allowed

@dataclass
class JobSource:
    """
    Defines policy for a job data source.
    
    Fields:
        source_id: Unique identifier for the source
        source_name: Human-readable name
        source_type: How data is fetched (api/rss/scraping/manual)
        allowed_mode: What level of data can be displayed publicly
        attribution_required: Whether source must be shown to users
        rate_limit: Rate limiting configuration
        takedown_contact: Email or URL for DMCA/takedown requests
        source_url: Base URL of the source
        terms_url: URL to the source's terms of service
        notes: Additional compliance notes
    """
    source_id: str
    source_name: str
    source_type: SourceType
    allowed_mode: AllowedMode
    attribution_required: bool
    rate_limit: RateLimit
    takedown_contact: str
    source_url: str = ""
    terms_url: str = ""
    notes: str = ""

# ═══════════════════════════════════════════════════════════════════════════
# JOB SOURCES REGISTRY (Configuration Data)
# ═══════════════════════════════════════════════════════════════════════════

JOB_SOURCES: Dict[str, JobSource] = {
    # ─────────────────────────────────────────────────────────────────────
    # API-BASED SOURCES (Generally Safer)
    # ─────────────────────────────────────────────────────────────────────
    
    "Greenhouse": JobSource(
        source_id="greenhouse",
        source_name="Greenhouse",
        source_type=SourceType.OFFICIAL_API,
        allowed_mode=AllowedMode.FULL_DISPLAY,
        attribution_required=False,
        rate_limit=RateLimit(requests_per_minute=60, burst=10),
        takedown_contact="support@greenhouse.io",
        source_url="https://boards.greenhouse.io",
        terms_url="https://www.greenhouse.io/terms-of-service",
        notes="Public job board API. B2B terms apply to employers, not aggregators."
    ),
    
    "Lever": JobSource(
        source_id="lever",
        source_name="Lever",
        source_type=SourceType.OFFICIAL_API,
        allowed_mode=AllowedMode.FULL_DISPLAY,
        attribution_required=False,
        rate_limit=RateLimit(requests_per_minute=60, burst=10),
        takedown_contact="privacy@lever.co",
        source_url="https://jobs.lever.co",
        terms_url="https://www.lever.co/terms-of-service",
        notes="Public job board API. No explicit prohibition on aggregation."
    ),
    
    "Workable": JobSource(
        source_id="workable",
        source_name="Workable",
        source_type=SourceType.OFFICIAL_API,
        allowed_mode=AllowedMode.FULL_DISPLAY,
        attribution_required=False,
        rate_limit=RateLimit(requests_per_minute=60, burst=10),
        takedown_contact="privacy@workable.com",
        source_url="https://apply.workable.com",
        terms_url="https://www.workable.com/terms",
        notes="Public job board API."
    ),
    
    "Jooble": JobSource(
        source_id="jooble",
        source_name="Jooble",
        source_type=SourceType.OFFICIAL_API,
        allowed_mode=AllowedMode.FULL_DISPLAY,
        attribution_required=False,
        rate_limit=RateLimit(requests_per_minute=30, burst=5),
        takedown_contact="support@jooble.org",
        source_url="https://jooble.org",
        terms_url="https://jooble.org/info/terms-of-use",
        notes="Partner API with API key. Full rights granted via API access."
    ),
    
    "RemoteOK": JobSource(
        source_id="remoteok",
        source_name="RemoteOK",
        source_type=SourceType.OFFICIAL_API,
        allowed_mode=AllowedMode.LIMITED_DISPLAY,
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=10, burst=2),
        takedown_contact="pieter@levels.io",
        source_url="https://remoteok.com",
        terms_url="https://remoteok.com/legal",
        notes="Public API available. Attribution required. Use limited display to be safe."
    ),
    
    "Remotive": JobSource(
        source_id="remotive",
        source_name="Remotive",
        source_type=SourceType.OFFICIAL_API,
        allowed_mode=AllowedMode.LIMITED_DISPLAY,
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=30, burst=5),
        takedown_contact="hello@remotive.io",
        source_url="https://remotive.io",
        terms_url="https://remotive.io/terms",
        notes="Public API. Attribution required per terms."
    ),
    
    "Himalayas": JobSource(
        source_id="himalayas",
        source_name="Himalayas",
        source_type=SourceType.OFFICIAL_API,
        allowed_mode=AllowedMode.LINK_ONLY,  # ⚠️ Terms prohibit commercial display
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=20, burst=3),
        takedown_contact="hello@himalayas.app",
        source_url="https://himalayas.app",
        terms_url="https://himalayas.app/terms",
        notes="⚠️ Terms Section 2 & 30 prohibit scraping and commercial use. LINK_ONLY recommended."
    ),
    
    "Jobicy": JobSource(
        source_id="jobicy",
        source_name="Jobicy",
        source_type=SourceType.OFFICIAL_API,
        allowed_mode=AllowedMode.LIMITED_DISPLAY,
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=20, burst=3),
        takedown_contact="hello@jobicy.com",
        source_url="https://jobicy.com",
        terms_url="https://jobicy.com/terms",
        notes="Public API with RSS. Attribution required."
    ),
    
    # ─────────────────────────────────────────────────────────────────────
    # SCRAPING SOURCES (Require More Caution)
    # ─────────────────────────────────────────────────────────────────────
    
    "Wuzzuf": JobSource(
        source_id="wuzzuf",
        source_name="Wuzzuf",
        source_type=SourceType.SCRAPING,
        allowed_mode=AllowedMode.LIMITED_DISPLAY,
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=10, burst=2),
        takedown_contact="info@wuzzuf.net",
        source_url="https://wuzzuf.net",
        terms_url="https://wuzzuf.net/terms-and-conditions",
        notes="Egypt's largest job board. Scraping with rate limits. Attribution recommended."
    ),
    
    "Forasna": JobSource(
        source_id="forasna",
        source_name="Forasna",
        source_type=SourceType.SCRAPING,
        allowed_mode=AllowedMode.LINK_ONLY,  # Default to safe mode - terms unclear
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=5, burst=1),
        takedown_contact="info@forasna.com",
        source_url="https://forasna.com",
        terms_url="https://forasna.com/terms",
        notes="Egyptian job board. Terms not verified - using LINK_ONLY as default."
    ),
    
    "Indeed Egypt": JobSource(
        source_id="indeed_egypt",
        source_name="Indeed Egypt",
        source_type=SourceType.SCRAPING,
        allowed_mode=AllowedMode.DISABLED,  # ❌ Terms explicitly prohibit scraping
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=0, burst=0),
        takedown_contact="legal@indeed.com",
        source_url="https://eg.indeed.com",
        terms_url="https://www.indeed.com/legal",
        notes="⛔ PERMANENTLY DISABLED - Terms explicitly forbid scraping. Not halal to use without permission."
    ),
    
    "Bayt.com": JobSource(
        source_id="bayt",
        source_name="Bayt.com",
        source_type=SourceType.SCRAPING,
        allowed_mode=AllowedMode.DISABLED,  # ⛔ Terms unclear - not halal to use
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=0, burst=0),
        takedown_contact="support@bayt.com",
        source_url="https://www.bayt.com",
        terms_url="https://www.bayt.com/en/terms/",
        notes="⛔ PERMANENTLY DISABLED - Terms unclear. Not halal to scrape without explicit permission."
    ),
    
    "LinkedIn": JobSource(
        source_id="linkedin",
        source_name="LinkedIn",
        source_type=SourceType.SCRAPING,
        allowed_mode=AllowedMode.DISABLED,  # ⛔ Terms explicitly prohibit scraping
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=0, burst=0),
        takedown_contact="copyright@linkedin.com",
        source_url="https://www.linkedin.com/jobs",
        terms_url="https://www.linkedin.com/legal/user-agreement",
        notes="⛔ PERMANENTLY DISABLED - Section 8.2.2 & 8.2.4 prohibit scraping. Not halal - violates their agreement."
    ),
    
    "Jobzella": JobSource(
        source_id="jobzella",
        source_name="Jobzella",
        source_type=SourceType.SCRAPING,
        allowed_mode=AllowedMode.DISABLED,  # ⛔ Terms unclear - not halal to use
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=0, burst=0),
        takedown_contact="support@jobzella.com",
        source_url="https://www.jobzella.com",
        terms_url="https://www.jobzella.com/terms",
        notes="⛔ PERMANENTLY DISABLED - Terms unclear. Not halal to scrape without explicit permission."
    ),
    
    "Tanqeeb": JobSource(
        source_id="tanqeeb",
        source_name="Tanqeeb",
        source_type=SourceType.SCRAPING,
        allowed_mode=AllowedMode.LINK_ONLY,  # Terms unclear
        attribution_required=True,
        rate_limit=RateLimit(requests_per_minute=5, burst=1),
        takedown_contact="info@tanqeeb.com",
        source_url="https://www.tanqeeb.com",
        terms_url="https://www.tanqeeb.com/terms",
        notes="Arab region job aggregator. Terms not verified - using LINK_ONLY."
    ),
}

# ═══════════════════════════════════════════════════════════════════════════
# DEFAULT POLICY FOR UNKNOWN SOURCES
# ═══════════════════════════════════════════════════════════════════════════

DEFAULT_SOURCE_POLICY = JobSource(
    source_id="unknown",
    source_name="Unknown Source",
    source_type=SourceType.MANUAL,
    allowed_mode=AllowedMode.LINK_ONLY,  # Safe default
    attribution_required=True,
    rate_limit=RateLimit(requests_per_minute=5, burst=1),
    takedown_contact="",
    source_url="",
    terms_url="",
    notes="Unknown source - using safest LINK_ONLY mode by default."
)

def get_source_policy(source_name: str) -> JobSource:
    """Get the policy for a source, returning default for unknown sources."""
    return JOB_SOURCES.get(source_name, DEFAULT_SOURCE_POLICY)

# ═══════════════════════════════════════════════════════════════════════════
# VALIDATOR FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def validate_source(source: JobSource) -> List[str]:
    """
    Validate a JobSource configuration for errors.
    Returns list of error messages (empty if valid).
    """
    errors = []
    
    # Check enum values are valid
    if not isinstance(source.source_type, SourceType):
        errors.append(f"Invalid source_type: {source.source_type}")
    
    if not isinstance(source.allowed_mode, AllowedMode):
        errors.append(f"Invalid allowed_mode: {source.allowed_mode}")
    
    # Check takedown_contact for non-disabled sources
    if source.allowed_mode != AllowedMode.DISABLED and not source.takedown_contact:
        errors.append(f"Missing takedown_contact for enabled source: {source.source_name}")
    
    # Check invalid combinations
    if source.source_type == SourceType.SCRAPING and source.allowed_mode == AllowedMode.FULL_DISPLAY:
        errors.append(f"Warning: Scraping source '{source.source_name}' set to FULL_DISPLAY - consider LIMITED_DISPLAY")
    
    # Check rate limits for non-disabled sources
    if source.allowed_mode != AllowedMode.DISABLED:
        if source.rate_limit.requests_per_minute <= 0:
            errors.append(f"Invalid rate_limit for enabled source: {source.source_name}")
    
    # Check attribution for scraped sources
    if source.source_type == SourceType.SCRAPING and not source.attribution_required:
        errors.append(f"Warning: Scraping source '{source.source_name}' should have attribution_required=True")
    
    return errors

def validate_all_sources() -> Dict[str, List[str]]:
    """Validate all configured sources. Returns dict of source_name -> errors."""
    results = {}
    for name, source in JOB_SOURCES.items():
        errors = validate_source(source)
        if errors:
            results[name] = errors
    return results

# Run validation on load
print("═" * 80)
print("📋 JOB SOURCES POLICY CONFIGURATION")
print("═" * 80)

# Summary stats
api_sources = [s for s in JOB_SOURCES.values() if s.source_type == SourceType.OFFICIAL_API]
scraping_sources = [s for s in JOB_SOURCES.values() if s.source_type == SourceType.SCRAPING]
disabled_sources = [s for s in JOB_SOURCES.values() if s.allowed_mode == AllowedMode.DISABLED]
full_display = [s for s in JOB_SOURCES.values() if s.allowed_mode == AllowedMode.FULL_DISPLAY]
limited_display = [s for s in JOB_SOURCES.values() if s.allowed_mode == AllowedMode.LIMITED_DISPLAY]
link_only = [s for s in JOB_SOURCES.values() if s.allowed_mode == AllowedMode.LINK_ONLY]

print(f"\n📊 Source Types:")
print(f"   API Sources: {len(api_sources)}")
print(f"   Scraping Sources: {len(scraping_sources)}")

print(f"\n🔒 Display Modes:")
print(f"   ✅ full_display: {len(full_display)} ({', '.join(s.source_name for s in full_display)})")
print(f"   ⚠️ limited_display: {len(limited_display)} ({', '.join(s.source_name for s in limited_display)})")
print(f"   🔗 link_only: {len(link_only)} ({', '.join(s.source_name for s in link_only)})")
print(f"   ❌ disabled: {len(disabled_sources)} ({', '.join(s.source_name for s in disabled_sources)})")

# Validate and show warnings
validation_results = validate_all_sources()
if validation_results:
    print(f"\n⚠️ Validation Warnings:")
    for source_name, errors in validation_results.items():
        for error in errors:
            print(f"   - {error}")
else:
    print(f"\n✅ All source configurations validated successfully!")

print(f"\n📝 Default Policy for Unknown Sources: {DEFAULT_SOURCE_POLICY.allowed_mode.value}")
print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
📋 JOB SOURCES POLICY CONFIGURATION
════════════════════════════════════════════════════════════════════════════════

📊 Source Types:
   API Sources: 8
   Scraping Sources: 7

🔒 Display Modes:
   ✅ full_display: 4 (Greenhouse, Lever, Workable, Jooble)
   ⚠️ limited_display: 4 (RemoteOK, Remotive, Jobicy, Wuzzuf)
   🔗 link_only: 3 (Himalayas, Forasna, Tanqeeb)
   ❌ disabled: 4 (Indeed Egypt, Bayt.com, LinkedIn, Jobzella)

✅ All source configurations validated successfully!

📝 Default Policy for Unknown Sources: link_only
════════════════════════════════════════════════════════════════════════════════


In [7]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                    API SERIALIZER & RATE LIMITER
# ════════════════════════════════════════════════════════════════════════════════════════
# Implements enforcement rules for display modes and per-source rate limiting.
# ════════════════════════════════════════════════════════════════════════════════════════

import time
from threading import Lock
from collections import defaultdict

# ═══════════════════════════════════════════════════════════════════════════
# RATE LIMITER (Per-Source)
# ═══════════════════════════════════════════════════════════════════════════

class SourceRateLimiter:
    """
    Token bucket rate limiter for per-source request limiting.
    Configurable without code changes via JOB_SOURCES registry.
    """
    
    def __init__(self):
        self._tokens: Dict[str, float] = defaultdict(float)
        self._last_update: Dict[str, float] = defaultdict(float)
        self._lock = Lock()
    
    def _get_source_limits(self, source_name: str) -> Tuple[int, int]:
        """Get rate limits from source policy."""
        policy = get_source_policy(source_name)
        return policy.rate_limit.requests_per_minute, policy.rate_limit.burst
    
    def acquire(self, source_name: str, timeout: float = 30.0) -> bool:
        """
        Acquire a token for the given source. Blocks until token available or timeout.
        Returns True if acquired, False if source is disabled or timeout.
        """
        rpm, burst = self._get_source_limits(source_name)
        
        # Disabled sources cannot acquire tokens
        policy = get_source_policy(source_name)
        if policy.allowed_mode == AllowedMode.DISABLED:
            return False
        
        if rpm <= 0:
            return False
        
        tokens_per_second = rpm / 60.0
        start_time = time.time()
        
        while True:
            with self._lock:
                current_time = time.time()
                source_key = source_name
                
                # Initialize if first request
                if source_key not in self._last_update or self._last_update[source_key] == 0:
                    self._tokens[source_key] = float(burst)
                    self._last_update[source_key] = current_time
                
                # Replenish tokens based on time elapsed
                time_passed = current_time - self._last_update[source_key]
                self._tokens[source_key] = min(
                    float(burst),
                    self._tokens[source_key] + time_passed * tokens_per_second
                )
                self._last_update[source_key] = current_time
                
                # Try to acquire token
                if self._tokens[source_key] >= 1.0:
                    self._tokens[source_key] -= 1.0
                    return True
            
            # Check timeout
            if time.time() - start_time >= timeout:
                return False
            
            # Wait before retry
            time.sleep(0.1)
    
    def get_wait_time(self, source_name: str) -> float:
        """Get estimated wait time in seconds for next available token."""
        rpm, burst = self._get_source_limits(source_name)
        if rpm <= 0:
            return float('inf')
        
        with self._lock:
            current_tokens = self._tokens.get(source_name, float(burst))
            if current_tokens >= 1.0:
                return 0.0
            
            tokens_per_second = rpm / 60.0
            tokens_needed = 1.0 - current_tokens
            return tokens_needed / tokens_per_second

# Global rate limiter instance
RATE_LIMITER = SourceRateLimiter()

# ═══════════════════════════════════════════════════════════════════════════
# API SERIALIZER (Shapes Job Payload Based on allowed_mode)
# ═══════════════════════════════════════════════════════════════════════════

class JobSerializer:
    """
    Serializes job records based on source policy.
    Enforces display mode restrictions.
    """
    
    # Maximum snippet length for limited_display mode
    SNIPPET_MAX_LENGTH = 200
    
    # Fields allowed for each display mode
    FULL_DISPLAY_FIELDS = [
        "job_id", "source_id", "source_name", "source_url",
        "title", "company", "location", "date", "description",
        "salary", "experience_years", "skills", "level",
        "work_type", "country", "city", "apply_url",
        "attribution_required"
    ]
    
    LIMITED_DISPLAY_FIELDS = [
        "job_id", "source_id", "source_name", "source_url",
        "title", "company", "location", "date", "description_snippet",
        "salary", "level", "work_type", "country", "city",
        "apply_url", "attribution_required"
    ]
    
    LINK_ONLY_FIELDS = [
        "job_id", "source_id", "source_name", "source_url",
        "title", "company", "location", "date",
        "apply_url", "attribution_required"
    ]
    
    @classmethod
    def _truncate_description(cls, text: str, max_length: int = None) -> str:
        """Truncate text to max length with ellipsis."""
        if max_length is None:
            max_length = cls.SNIPPET_MAX_LENGTH
        
        if not text or len(text) <= max_length:
            return text
        
        # Truncate at word boundary
        truncated = text[:max_length].rsplit(' ', 1)[0]
        return truncated.rstrip('.,!?;:') + "..."
    
    @classmethod
    def serialize_job(cls, job: Dict[str, Any], source_name: str = None) -> Optional[Dict[str, Any]]:
        """
        Serialize a job record according to its source policy.
        
        Args:
            job: Raw job dictionary
            source_name: Source name (uses job['source'] if not provided)
            
        Returns:
            Serialized job dict or None if source is disabled
        """
        source = source_name or job.get("source", "")
        policy = get_source_policy(source)
        
        # DISABLED: Exclude from all public results
        if policy.allowed_mode == AllowedMode.DISABLED:
            return None
        
        # Base job data with source metadata
        serialized = {
            "job_id": job.get("job_id", hash(f"{job.get('title', '')}{job.get('company', '')}{job.get('url', '')}")),
            "source_id": policy.source_id,
            "source_name": policy.source_name if policy.attribution_required else None,
            "source_url": policy.source_url if policy.attribution_required else None,
            "attribution_required": policy.attribution_required,
            "display_mode": policy.allowed_mode.value,
        }
        
        # LINK_ONLY: Only minimal card fields + apply URL
        if policy.allowed_mode == AllowedMode.LINK_ONLY:
            serialized.update({
                "title": job.get("title", ""),
                "company": job.get("company", ""),
                "location": job.get("location", ""),
                "date": job.get("date", ""),
                "apply_url": job.get("url", ""),
            })
            return serialized
        
        # LIMITED_DISPLAY: Metadata + snippet only
        if policy.allowed_mode == AllowedMode.LIMITED_DISPLAY:
            description = job.get("description", "")
            serialized.update({
                "title": job.get("title", ""),
                "company": job.get("company", ""),
                "location": job.get("location", ""),
                "date": job.get("date", ""),
                "description_snippet": cls._truncate_description(description),
                "salary": job.get("salary", ""),
                "level": job.get("level", ""),
                "work_type": job.get("work_type", ""),
                "country": job.get("country", ""),
                "city": job.get("city", ""),
                "apply_url": job.get("url", ""),
            })
            return serialized
        
        # FULL_DISPLAY: All fields allowed
        if policy.allowed_mode == AllowedMode.FULL_DISPLAY:
            serialized.update({
                "title": job.get("title", ""),
                "company": job.get("company", ""),
                "location": job.get("location", ""),
                "date": job.get("date", ""),
                "description": job.get("description", ""),
                "salary": job.get("salary", ""),
                "experience_years": job.get("experience_years", ""),
                "skills": job.get("skills", ""),
                "level": job.get("level", ""),
                "work_type": job.get("work_type", ""),
                "country": job.get("country", ""),
                "city": job.get("city", ""),
                "apply_url": job.get("url", ""),
            })
            return serialized
        
        return serialized
    
    @classmethod
    def serialize_jobs_list(cls, jobs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Serialize a list of jobs, filtering out disabled sources.
        
        Args:
            jobs: List of raw job dictionaries
            
        Returns:
            List of serialized jobs (disabled sources excluded)
        """
        serialized = []
        for job in jobs:
            result = cls.serialize_job(job)
            if result is not None:
                serialized.append(result)
        return serialized
    
    @classmethod
    def get_api_response(cls, jobs: List[Dict[str, Any]], include_meta: bool = True) -> Dict[str, Any]:
        """
        Generate a full API response with metadata.
        
        Args:
            jobs: List of raw job dictionaries
            include_meta: Whether to include response metadata
            
        Returns:
            API response dictionary
        """
        serialized_jobs = cls.serialize_jobs_list(jobs)
        
        response = {
            "success": True,
            "count": len(serialized_jobs),
            "jobs": serialized_jobs,
        }
        
        if include_meta:
            # Count by display mode
            mode_counts = defaultdict(int)
            attribution_count = 0
            
            for job in serialized_jobs:
                mode_counts[job.get("display_mode", "unknown")] += 1
                if job.get("attribution_required"):
                    attribution_count += 1
            
            response["meta"] = {
                "display_modes": dict(mode_counts),
                "attribution_required_count": attribution_count,
                "disabled_sources": [
                    s.source_name for s in JOB_SOURCES.values() 
                    if s.allowed_mode == AllowedMode.DISABLED
                ],
            }
        
        return response

# ═══════════════════════════════════════════════════════════════════════════
# ATTRIBUTION HELPER
# ═══════════════════════════════════════════════════════════════════════════

def get_attribution_text(source_name: str) -> Optional[str]:
    """
    Get attribution text for a source if required.
    Returns None if attribution not required.
    """
    policy = get_source_policy(source_name)
    if not policy.attribution_required:
        return None
    
    if policy.source_url:
        return f"Source: {policy.source_name} ({policy.source_url})"
    return f"Source: {policy.source_name}"

def must_show_attribution(source_name: str) -> bool:
    """Check if attribution must be shown for a source."""
    policy = get_source_policy(source_name)
    return policy.attribution_required

# ═══════════════════════════════════════════════════════════════════════════
# USAGE EXAMPLES
# ═══════════════════════════════════════════════════════════════════════════

print("═" * 80)
print("🔧 API SERIALIZER & RATE LIMITER LOADED")
print("═" * 80)

print("\n📋 Example Usage:")
print("""
# Rate limiting before fetch:
if RATE_LIMITER.acquire("Wuzzuf"):
    response = requests.get(url)
else:
    print("Rate limited or source disabled")

# Serialize jobs for API response:
api_response = JobSerializer.get_api_response(jobs_list)

# Check display mode for a source:
policy = get_source_policy("LinkedIn")
print(f"LinkedIn mode: {policy.allowed_mode.value}")  # disabled

# Get attribution text:
attribution = get_attribution_text("RemoteOK")
print(attribution)  # "Source: RemoteOK (https://remoteok.com)"
""")

print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
🔧 API SERIALIZER & RATE LIMITER LOADED
════════════════════════════════════════════════════════════════════════════════

📋 Example Usage:

# Rate limiting before fetch:
if RATE_LIMITER.acquire("Wuzzuf"):
    response = requests.get(url)
else:
    print("Rate limited or source disabled")

# Serialize jobs for API response:
api_response = JobSerializer.get_api_response(jobs_list)

# Check display mode for a source:
policy = get_source_policy("LinkedIn")
print(f"LinkedIn mode: {policy.allowed_mode.value}")  # disabled

# Get attribution text:
attribution = get_attribution_text("RemoteOK")
print(attribution)  # "Source: RemoteOK (https://remoteok.com)"

════════════════════════════════════════════════════════════════════════════════


In [8]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                    CONFIGURATION FILE I/O (Configurable Without Code Changes)
# ════════════════════════════════════════════════════════════════════════════════════════
# Export/Import job_sources configuration to/from JSON for easy editing.
# ════════════════════════════════════════════════════════════════════════════════════════

import os

CONFIG_FILE_PATH = "job_sources_config.json"

def export_sources_config(filepath: str = CONFIG_FILE_PATH) -> None:
    """
    Export all job source configurations to a JSON file.
    This allows editing policies without code changes.
    """
    config = {
        "version": "1.0",
        "generated_at": datetime.now().isoformat(),
        "description": "Job source policy configuration. Edit this file to change source policies without code changes.",
        "enums": {
            "source_type": ["official_api", "rss_feed", "scraping", "manual"],
            "allowed_mode": ["full_display", "limited_display", "link_only", "disabled"],
        },
        "enforcement_rules": {
            "full_display": "API can return full job details including description",
            "limited_display": "Return metadata + snippet only (max 200 chars), always include apply_url",
            "link_only": "Return ONLY minimal fields (title/company/location/date) + apply_url",
            "disabled": "Exclude from all public results",
        },
        "default_policy": {
            "source_id": DEFAULT_SOURCE_POLICY.source_id,
            "source_name": DEFAULT_SOURCE_POLICY.source_name,
            "source_type": DEFAULT_SOURCE_POLICY.source_type.value,
            "allowed_mode": DEFAULT_SOURCE_POLICY.allowed_mode.value,
            "attribution_required": DEFAULT_SOURCE_POLICY.attribution_required,
            "rate_limit": {
                "requests_per_minute": DEFAULT_SOURCE_POLICY.rate_limit.requests_per_minute,
                "burst": DEFAULT_SOURCE_POLICY.rate_limit.burst,
            },
            "takedown_contact": DEFAULT_SOURCE_POLICY.takedown_contact,
            "source_url": DEFAULT_SOURCE_POLICY.source_url,
            "terms_url": DEFAULT_SOURCE_POLICY.terms_url,
            "notes": DEFAULT_SOURCE_POLICY.notes,
        },
        "sources": {}
    }
    
    for name, source in JOB_SOURCES.items():
        config["sources"][name] = {
            "source_id": source.source_id,
            "source_name": source.source_name,
            "source_type": source.source_type.value,
            "allowed_mode": source.allowed_mode.value,
            "attribution_required": source.attribution_required,
            "rate_limit": {
                "requests_per_minute": source.rate_limit.requests_per_minute,
                "burst": source.rate_limit.burst,
            },
            "takedown_contact": source.takedown_contact,
            "source_url": source.source_url,
            "terms_url": source.terms_url,
            "notes": source.notes,
        }
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=2)
    
    print(f"✅ Configuration exported to: {filepath}")


def import_sources_config(filepath: str = CONFIG_FILE_PATH) -> bool:
    """
    Import job source configurations from a JSON file.
    Returns True if successful, False otherwise.
    """
    global JOB_SOURCES, DEFAULT_SOURCE_POLICY
    
    if not os.path.exists(filepath):
        print(f"⚠️ Config file not found: {filepath}")
        return False
    
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            config = json.load(f)
        
        # Import sources
        for name, data in config.get("sources", {}).items():
            JOB_SOURCES[name] = JobSource(
                source_id=data.get("source_id", name.lower().replace(" ", "_")),
                source_name=data.get("source_name", name),
                source_type=SourceType(data.get("source_type", "manual")),
                allowed_mode=AllowedMode(data.get("allowed_mode", "link_only")),
                attribution_required=data.get("attribution_required", True),
                rate_limit=RateLimit(
                    requests_per_minute=data.get("rate_limit", {}).get("requests_per_minute", 5),
                    burst=data.get("rate_limit", {}).get("burst", 1),
                ),
                takedown_contact=data.get("takedown_contact", ""),
                source_url=data.get("source_url", ""),
                terms_url=data.get("terms_url", ""),
                notes=data.get("notes", ""),
            )
        
        print(f"✅ Configuration imported from: {filepath}")
        print(f"   Loaded {len(config.get('sources', {}))} sources")
        return True
        
    except Exception as e:
        print(f"❌ Error importing config: {e}")
        return False


def get_sources_summary() -> pd.DataFrame:
    """Get a summary DataFrame of all source policies."""
    data = []
    for name, source in JOB_SOURCES.items():
        data.append({
            "Source": name,
            "Source_ID": source.source_id,
            "Type": source.source_type.value,
            "Mode": source.allowed_mode.value,
            "Attribution": "Yes" if source.attribution_required else "No",
            "Rate_Limit": f"{source.rate_limit.requests_per_minute}/min",
            "Takedown_Contact": source.takedown_contact or "N/A",
        })
    return pd.DataFrame(data)


# ═══════════════════════════════════════════════════════════════════════════
# EXPORT CONFIGURATION ON LOAD
# ═══════════════════════════════════════════════════════════════════════════

print("═" * 80)
print("📁 CONFIGURATION FILE I/O")
print("═" * 80)

# Export config file for easy editing
export_sources_config()

print("\n📋 Source Policies Summary:")
print(get_sources_summary().to_string(index=False))

print("\n💡 To modify policies without code changes:")
print(f"   1. Edit the file: {CONFIG_FILE_PATH}")
print("   2. Run: import_sources_config()")
print("   3. Re-run the aggregator")
print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
📁 CONFIGURATION FILE I/O
════════════════════════════════════════════════════════════════════════════════
✅ Configuration exported to: job_sources_config.json

📋 Source Policies Summary:
      Source    Source_ID         Type            Mode Attribution Rate_Limit       Takedown_Contact
  Greenhouse   greenhouse official_api    full_display          No     60/min  support@greenhouse.io
       Lever        lever official_api    full_display          No     60/min       privacy@lever.co
    Workable     workable official_api    full_display          No     60/min   privacy@workable.com
      Jooble       jooble official_api    full_display          No     30/min     support@jooble.org
    RemoteOK     remoteok official_api limited_display         Yes     10/min       pieter@levels.io
    Remotive     remotive official_api limited_display         Yes     30/min      hello@remotive.io
   Himalayas    himalayas

In [9]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              KEYWORDS & LOCATION DATA
# ════════════════════════════════════════════════════════════════════════════════════════

# Tech job keywords (INCLUDE if title contains any)
TECH_KEYWORDS = [
    # Database & SQL Development (Priority)
    "oracle", "pl/sql", "plsql", "sql developer", "database developer", "db developer",
    "database engineer", "data engineer", "etl", "data warehouse", "dwh",
    "postgresql", "mysql", "sql server", "mssql", "t-sql", "tsql",
    "apex", "oracle forms", "oracle reports", "informatica", "talend", "ssis",
    "power bi", "tableau", "bi developer", "report developer", "data modeling",
    
    # General Software Engineering
    "software", "engineer", "engineering", "developer", "programmer", "coder",
    "backend", "back-end", "frontend", "front-end", "fullstack", "full-stack",
    "mobile", "android", "ios", "flutter", "react native", "swift", "kotlin",
    "platform", "embedded", "firmware", "systems", "devops", "devsecops", "sre",
    "site reliability", "infrastructure", "cloud", "aws", "azure", "gcp",
    "kubernetes", "docker", "terraform", "linux", "sysadmin", "network engineer",
    "data scientist", "data analyst", "analytics",
    "machine learning", "ml engineer", "ai engineer", "nlp", "deep learning",
    "qa", "quality assurance", "test engineer", "automation engineer", "sdet",
    "security", "cybersecurity", "infosec", "soc analyst", "appsec",
    "database", "dba", "python", "java developer", "golang",
    "node", "nodejs", "javascript", "typescript", "react", "angular", "vue",
    ".net", "dotnet", "c#", "php", "ruby", "rails", "rust", "scala",
    "technical support", "it support", "tech support",
]

# Non-tech keywords (EXCLUDE if title contains any)
NON_TECH_KEYWORDS = [
    "sales", "telesales", "account manager", "account executive", "business development",
    "marketing", "content writer", "copywriter", "social media", "community manager",
    "hr", "human resources", "recruiter", "talent acquisition", "customer support",
    "customer service", "call center", "operations manager", "logistics", "warehouse",
    "finance", "accountant", "legal", "admin", "administrative", "receptionist",
    "nurse", "pharmacist", "doctor", "chef", "driver", "delivery",
]

if INCLUDE_PRODUCT:
    TECH_KEYWORDS += ["product manager", "product owner", "business analyst", 
                      "systems analyst", "solution architect", "enterprise architect"]
if INCLUDE_DESIGN:
    TECH_KEYWORDS += ["ui", "ux", "ui/ux", "user experience", "product designer"]

# ════════════════════════════════════════════════════════════════════════════════════════
#                      EGYPT CITIES/GOVERNORATES MAPPING
# ════════════════════════════════════════════════════════════════════════════════════════

# Map keywords to their city/governorate name
EGYPT_CITY_MAPPING = {
    # Cairo Governorate
    "cairo": "Cairo", "القاهرة": "Cairo", "nasr city": "Cairo", "مدينة نصر": "Cairo",
    "heliopolis": "Cairo", "مصر الجديدة": "Cairo", "maadi": "Cairo", "المعادي": "Cairo",
    "zamalek": "Cairo", "الزمالك": "Cairo", "dokki": "Cairo", "الدقي": "Cairo",
    "mohandessin": "Cairo", "المهندسين": "Cairo", "downtown": "Cairo",
    "garden city": "Cairo", "new cairo": "New Cairo", "التجمع": "New Cairo",
    "tagamoa": "New Cairo", "fifth settlement": "New Cairo", "التجمع الخامس": "New Cairo",
    "rehab": "New Cairo", "الرحاب": "New Cairo", "shorouk": "New Cairo",
    
    # Giza Governorate
    "giza": "Giza", "الجيزة": "Giza", "6th of october": "6th of October City",
    "sixth of october": "6th of October City", "october city": "6th of October City",
    "٦ أكتوبر": "6th of October City", "6 أكتوبر": "6th of October City",
    "sheikh zayed": "Sheikh Zayed City", "الشيخ زايد": "Sheikh Zayed City",
    "smart village": "Smart Village", "القرية الذكية": "Smart Village",
    "hadayek october": "Giza", "dream land": "Giza", "beverly hills": "Giza",
    
    # Alexandria Governorate
    "alexandria": "Alexandria", "alex": "Alexandria", "الإسكندرية": "Alexandria",
    "الاسكندرية": "Alexandria",
    
    # Other Governorates
    "mansoura": "Dakahlia", "المنصورة": "Dakahlia",
    "tanta": "Gharbia", "طنطا": "Gharbia",
    "damietta": "Damietta", "دمياط": "Damietta",
    "ismailia": "Ismailia", "الإسماعيلية": "Ismailia",
    "suez": "Suez", "السويس": "Suez",
    "port said": "Port Said", "portsaid": "Port Said", "بورسعيد": "Port Said",
    "assiut": "Assiut", "asyut": "Assiut", "أسيوط": "Assiut",
    "aswan": "Aswan", "أسوان": "Aswan",
    "fayoum": "Fayoum", "الفيوم": "Fayoum",
    "zagazig": "Sharqia", "الزقازيق": "Sharqia",
    "banha": "Qalyubia", "بنها": "Qalyubia",
    "qena": "Qena", "قنا": "Qena",
    "luxor": "Luxor", "الأقصر": "Luxor",
    "hurghada": "Red Sea", "الغردقة": "Red Sea",
    "sharm": "South Sinai", "sharm el sheikh": "South Sinai", "شرم الشيخ": "South Sinai",
    "el gouna": "Red Sea", "الجونة": "Red Sea",
    "minya": "Minya", "المنيا": "Minya",
    "sohag": "Sohag", "سوهاج": "Sohag",
    "new administrative capital": "New Administrative Capital", "nac": "New Administrative Capital",
    "العاصمة الإدارية": "New Administrative Capital",
    
    # Generic Egypt
    "egypt": "Egypt (General)", "مصر": "Egypt (General)",
}

# Remote keywords
REMOTE_KEYWORDS = ["remote", "work from home", "wfh", "anywhere", "distributed", 
                   "fully remote", "remote-first", "remotely", "عن بعد", "worldwide",
                   "work remotely", "home office", "hybrid"]

# Country detection keywords
COUNTRY_KEYWORDS = {
    "Egypt": ["egypt", "cairo", "giza", "alexandria", "new cairo", "nasr city", "heliopolis",
              "maadi", "zamalek", "dokki", "mohandessin", "rehab", "tagamoa", "fifth settlement",
              "6th of october", "sheikh zayed", "smart village", "mansoura", "tanta", "ismailia",
              "مصر", "القاهرة", "الجيزة", "الإسكندرية", "المعادي", "التجمع"],
    "UAE": ["uae", "dubai", "abu dhabi", "sharjah", "united arab emirates", "الإمارات", "دبي"],
    "Saudi Arabia": ["saudi", "ksa", "riyadh", "jeddah", "dammam", "السعودية", "الرياض"],
    "Qatar": ["qatar", "doha", "قطر"],
    "Kuwait": ["kuwait", "الكويت"],
    "Bahrain": ["bahrain", "manama", "البحرين"],
    "Jordan": ["jordan", "amman", "الأردن"],
    "Lebanon": ["lebanon", "beirut", "لبنان"],
    "Morocco": ["morocco", "casablanca", "المغرب"],
    "USA": ["usa", "united states", "america", "new york", "california", "texas"],
    "UK": ["uk", "united kingdom", "london", "england"],
    "Germany": ["germany", "berlin", "munich", "frankfurt"],
    "Netherlands": ["netherlands", "amsterdam", "holland"],
    "Canada": ["canada", "toronto", "vancouver", "montreal"],
    "Remote": ["remote", "worldwide", "anywhere", "distributed"],
}

# Known Egyptian companies (bypass location filter) - Expanded list
KNOWN_EGYPTIAN_COMPANIES = {
    # Tech Companies & Startups
    "sumerge", "advansys", "cequens", "integrant", "blabs", "blink22",
    "nawy", "robusta", "rubikal", "finaira", "egyptian banks company",
    "banque misr", "money fellows", "btech", "infomineo", "xenon7",
    "dsquares", "tagaddod", "dopay", "adree", "nowlun", "lawazem",
    "flat6labs", "bosta", "yassir", "soum", "econstruct", "careem",
    
    # Fintech & Payments
    "paymob", "fawry", "moneyhash", "kashier", "telda", "sympl", "valify",
    "khazna", "nowpay", "lucky", "opaay", "moneyfellows",
    
    # E-commerce & Marketplaces
    "jumia", "souq", "noon", "talabat", "elmenus", "breadfast", "rabbit",
    "homzmart", "brimore", "capiter", "maxab", "cartona", "kenzz",
    
    # Healthcare & Wellness
    "vezeeta", "yodawy", "chefaa", "almouneer", "shezlong",
    
    # Logistics & Delivery
    "trella", "swvl", "halan", "wasla", "weelo", "flextock",
    
    # SaaS & Enterprise
    "instabug", "synapse", "zammit", "elves", "localized", "sortino",
    "sylndr", "amenli", "nkhla", "geidea",
    
    # Telecom & Infrastructure
    "vodafone egypt", "orange egypt", "etisalat", "we telecom",
    "raya", "iti", "eitesal", "link development",
    
    # Banks & Financial Institutions
    "cib", "commercial international bank", "nbe", "national bank of egypt",
    "qnb", "hsbc egypt", "arab african international bank",
    
    # Consulting & Services
    "ejada", "atos egypt", "ibm egypt", "microsoft egypt", "dell egypt",
    "oracle egypt", "sap egypt", "accenture egypt",
}

print("✅ Keywords and location data loaded!")
print(f"   Known Egyptian companies: {len(KNOWN_EGYPTIAN_COMPANIES)}")

✅ Keywords and location data loaded!
   Known Egyptian companies: 97


In [10]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              COMPANY SOURCES
# ════════════════════════════════════════════════════════════════════════════════════════

# Greenhouse Boards (validated slugs) - Including companies with Egypt/MENA presence
GREENHOUSE_BOARDS = [
    # Companies with Egypt/MENA offices or remote-friendly
    "careem", "propertyfinder", "swvl", "paymob", "vezeeta", "instabug",
    
    # Global Tech Companies (Remote-friendly)
    "canonical", "mongodb", "elastic", "gitlab", "twilio", "datadog", 
    "stripe", "figma", "airtable", "asana", "dropbox", "intercom", 
    "mixpanel", "braze", "calendly", "typeform", "webflow", "vercel", 
    "planetscale", "cockroachlabs", "launchdarkly", "contentful", "algolia", 
    "cloudflare", "fastly", "okta", "pagerduty", "speechify", "udacity", 
    "coursera", "duolingo", "agoda", "trivago", "n26", "monzo", "marqeta", 
    "gusto", "remote", "lattice", "salesloft", "apollo", "deel", "oyster",
    "hubspot", "notion", "linear", "supabase", "dbt-labs", "airbyte", "mux",
]

# Lever Sites (CASE-SENSITIVE!) - Including Egypt-focused companies
LEVER_SITES = [
    # Egyptian Companies (exact case from their Lever URLs)
    "Bosta", "Yassir", "soum", "econstruct",
    
    # Global Remote-Friendly Companies  
    "welocalize", "rws", "aleph", "gradion", "toptal", "neon", 
    "metabase", "zerotier", "teleport", "secureframe", "sysdig",
]

# Workable Slugs - Egyptian Companies & Regional
WORKABLE_SLUGS = [
    # Egyptian Tech Companies & Startups
    "cequens", "integrant", "blabs", "blink22-3", "nawy-real-estate",
    "robusta", "rubikal", "sumerge-1", "finaira", "egyptian-banks-company-4",
    "bm-to", "money-fellows", "advansys-esc-1", "mylo-btech", "infomineo",
    "xenon7", "dsquares-loyalty-dmcc", "tagaddod", "dopay-8", "adree",
    "nowlun", "lawazem", "flat6labs", "covergo", "foodics", "gathern", 
    "dubizzlemena", "lucidya", "smarttechsa", "buildinglink",
    
    # Additional Egyptian/MENA Companies (verify slugs exist)
    "moneyhash", "kashier", "telda", "homzmart", "brimore", "sympl",
]

print(f"✅ Company Sources Loaded:")
print(f"   Greenhouse: {len(GREENHOUSE_BOARDS)} boards")
print(f"   Lever: {len(LEVER_SITES)} sites (CASE-SENSITIVE!)")
print(f"   Workable: {len(WORKABLE_SLUGS)} slugs")

✅ Company Sources Loaded:
   Greenhouse: 56 boards
   Lever: 15 sites (CASE-SENSITIVE!)
   Workable: 36 slugs


In [11]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              HELPER FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════════════════

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

def norm(s: str) -> str:
    """Normalize text for matching."""
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

def contains_any(text: str, keywords: List[str]) -> bool:
    """Check if normalized text contains any keyword."""
    t = norm(text)
    return any(k.lower() in t for k in keywords)

def detect_country(location: str, company: str = "") -> str:
    """Detect country from location string."""
    if not location:
        # Check if company is known Egyptian
        if company and any(known in norm(company) for known in KNOWN_EGYPTIAN_COMPANIES):
            return "Egypt"
        return "Unknown"
    
    loc_lower = location.lower()
    for country, keywords in COUNTRY_KEYWORDS.items():
        if any(kw in loc_lower for kw in keywords):
            return country
    return "Other"

def detect_egypt_city(location: str, company: str = "") -> str:
    """Detect Egypt city/governorate from location string."""
    if not location:
        # For known Egyptian companies without location, return general
        if company and any(known in norm(company) for known in KNOWN_EGYPTIAN_COMPANIES):
            return "Egypt (General)"
        return ""
    
    loc_lower = location.lower()
    
    # Check each city keyword
    for keyword, city in EGYPT_CITY_MAPPING.items():
        if keyword in loc_lower:
            return city
    
    return ""

def is_remote_job(location: str, title: str = "") -> str:
    """Detect if job is Remote, Hybrid, or On-site (for Egypt jobs)."""
    text = f"{location} {title}".lower()
    
    # Check for remote keywords
    if any(kw in text for kw in REMOTE_KEYWORDS):
        # Check if it's hybrid
        if "hybrid" in text:
            return "Hybrid"
        return "Remote"
    
    return "On-site"

def get_work_type_for_egyptians(location: str, title: str, company: str) -> str:
    """
    Get work type from an Egyptian perspective:
    - For Egypt jobs: Remote/Hybrid/On-site
    - For non-Egypt jobs: 'Remote' if accepts remote, 'N/A' if requires relocation
    """
    country = detect_country(location, company)
    text = f"{location} {title}".lower()
    
    if country == "Egypt":
        # For Egypt jobs, show actual work type
        if any(kw in text for kw in REMOTE_KEYWORDS):
            if "hybrid" in text:
                return "Hybrid"
            return "Remote"
        return "On-site"
    else:
        # For non-Egypt jobs, only show "Remote" if it accepts remote workers
        # Otherwise it's not applicable for Egyptians (requires relocation)
        if any(kw in text for kw in REMOTE_KEYWORDS):
            return "Remote"
        return "Relocation Required"

def is_known_egyptian_company(company: str) -> bool:
    """Check if company is in our known Egyptian companies whitelist."""
    if not company:
        return False
    c = norm(company)
    return any(known in c or c in known for known in KNOWN_EGYPTIAN_COMPANIES)

def is_egypt_location(loc: str, company: str = "") -> bool:
    """Check if location matches Egypt signals."""
    if is_known_egyptian_company(company):
        return True
    if not loc:
        return False
    country = detect_country(loc)
    if country == "Egypt":
        return True
    if INCLUDE_REMOTE_EGYPT and country == "Remote":
        return contains_any(loc, ["egypt", "مصر", "eg"])
    return False

def is_tech_job(title: str, dept: str = "") -> bool:
    """Check if job is tech-related."""
    title_n = norm(title)
    if contains_any(title_n, NON_TECH_KEYWORDS):
        return False
    if contains_any(title_n, TECH_KEYWORDS):
        return True
    blob = f"{title} {dept}"
    return contains_any(blob, TECH_KEYWORDS)

def detect_experience_level(title: str) -> str:
    """
    Detect experience level from job title.
    Returns: Senior, Mid, Junior, Lead, Principal, Intern
    """
    t = title.lower()
    
    # Leadership/Principal level (highest)
    if any(kw in t for kw in ["principal", "staff engineer", "distinguished", "fellow"]):
        return "Principal"
    
    # Lead/Manager level
    if any(kw in t for kw in ["lead", "team lead", "tech lead", "engineering manager", 
                               "head of", "director", "vp ", "vice president", "chief"]):
        return "Lead"
    
    # Senior level
    if any(kw in t for kw in ["senior", "sr.", "sr ", "ssr", "senior-level", 
                               "experienced", "expert"]):
        return "Senior"
    
    # Junior level
    if any(kw in t for kw in ["junior", "jr.", "jr ", "entry", "entry-level", 
                               "graduate", "fresh", "fresher", "trainee", 
                               "associate", "beginner"]):
        return "Junior"
    
    # Intern level
    if any(kw in t for kw in ["intern", "internship", "co-op", "student", "apprentice"]):
        return "Intern"
    
    # Mid-level indicators (explicit)
    if any(kw in t for kw in ["mid-level", "mid level", "intermediate", "ii", "iii", 
                               "level 2", "level 3", "l2", "l3"]):
        return "Mid"
    
    # Default: if no level specified, assume Mid-level
    return "Mid"

def extract_salary(text: str) -> str:
    """
    Extract salary information from job title, description, or metadata.
    Returns salary range string or empty string if not found.
    
    Supports formats:
    - $50,000 - $80,000
    - $50K - $80K
    - 50000 - 80000 USD
    - EGP 15,000 - 25,000
    - £40,000 - £60,000
    """
    if not text:
        return ""
    
    t = text.lower()
    
    # Common patterns for salary extraction
    patterns = [
        # USD formats
        r'\$[\d,]+(?:k)?\s*[-–to]+\s*\$[\d,]+(?:k)?(?:\s*(?:usd|per\s*year|annually|/yr|/year))?',
        r'\$[\d,]+(?:k)?(?:\s*(?:usd|per\s*year|annually|/yr|/year))?',
        r'[\d,]+\s*[-–to]+\s*[\d,]+\s*(?:usd|dollars)',
        
        # EUR/GBP formats
        r'€[\d,]+(?:k)?\s*[-–to]+\s*€[\d,]+(?:k)?',
        r'£[\d,]+(?:k)?\s*[-–to]+\s*£[\d,]+(?:k)?',
        
        # EGP formats (Egyptian Pound)
        r'egp\s*[\d,]+\s*[-–to]+\s*[\d,]+',
        r'[\d,]+\s*[-–to]+\s*[\d,]+\s*egp',
        r'le\s*[\d,]+\s*[-–to]+\s*[\d,]+',
        
        # Generic with K notation
        r'[\d]+k\s*[-–to]+\s*[\d]+k',
        
        # Salary keyword followed by amount
        r'salary[:\s]+[\d,]+\s*[-–to]+\s*[\d,]+',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, t, re.IGNORECASE)
        if match:
            salary = match.group(0).strip()
            # Normalize the format
            salary = salary.replace('–', '-').replace(' to ', '-')
            return salary.upper() if 'usd' in salary.lower() or 'egp' in salary.lower() else salary
    
    return ""

def extract_years_experience(text: str) -> str:
    """
    Extract years of experience requirement from job title or description.
    Returns experience string or empty if not found.
    
    Supports formats:
    - 3+ years
    - 3-5 years experience
    - minimum 2 years
    - at least 5 years
    """
    if not text:
        return ""
    
    t = text.lower()
    
    patterns = [
        # X+ years
        r'(\d+)\+?\s*(?:years?|yrs?)(?:\s*(?:of\s*)?(?:experience|exp))?',
        # X-Y years
        r'(\d+)\s*[-–to]+\s*(\d+)\s*(?:years?|yrs?)(?:\s*(?:of\s*)?(?:experience|exp))?',
        # minimum/at least X years
        r'(?:minimum|min|at\s*least)\s*(\d+)\s*(?:years?|yrs?)',
        # X years experience
        r'(\d+)\s*(?:years?|yrs?)\s*(?:of\s*)?(?:experience|exp)',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, t)
        if match:
            groups = match.groups()
            if len(groups) == 2 and groups[1]:  # Range like 3-5 years
                return f"{groups[0]}-{groups[1]} years"
            elif groups[0]:  # Single number
                return f"{groups[0]}+ years"
    
    return ""

def extract_skills(text: str) -> str:
    """
    Extract key technical skills mentioned in job title/description.
    Returns comma-separated skills or empty string.
    Uses word boundary matching to avoid false positives.
    """
    if not text:
        return ""
    
    t = text.lower()
    
    # Skills with their proper display format
    # Format: (search_pattern, display_name)
    skill_patterns = [
        # ═══════════════════════════════════════════════════════════════════════
        # DATABASE & SQL SKILLS (Priority)
        # ═══════════════════════════════════════════════════════════════════════
        (r'\boracle\b', "Oracle"),
        (r'\bpl/?sql\b|plsql\b', "PL/SQL"),
        (r'\bsql\s*developer\b', "SQL Developer"),
        (r'\bt-sql\b|tsql\b|transact-sql\b', "T-SQL"),
        (r'\bsql\s*server\b|mssql\b', "SQL Server"),
        (r'\bpostgresql\b|postgres\b', "PostgreSQL"),
        (r'\bmysql\b', "MySQL"),
        (r'\bmariadb\b', "MariaDB"),
        (r'\bsqlite\b', "SQLite"),
        (r'\bapex\b', "Oracle APEX"),
        (r'\boracle\s*forms\b', "Oracle Forms"),
        (r'\boracle\s*reports\b', "Oracle Reports"),
        (r'\betl\b', "ETL"),
        (r'\bssis\b', "SSIS"),
        (r'\bssrs\b', "SSRS"),
        (r'\bssas\b', "SSAS"),
        (r'\binformatica\b', "Informatica"),
        (r'\btalend\b', "Talend"),
        (r'\bdatastage\b', "DataStage"),
        (r'\bdata\s*warehouse\b|dwh\b', "Data Warehouse"),
        (r'\bdata\s*model', "Data Modeling"),
        (r'\bpower\s*bi\b|powerbi\b', "Power BI"),
        (r'\btableau\b', "Tableau"),
        (r'\blooker\b', "Looker"),
        (r'\bcognos\b', "Cognos"),
        (r'\bqlik\b', "Qlik"),
        (r'\bsnowflake\b', "Snowflake"),
        (r'\bredshift\b', "Redshift"),
        (r'\bbigquery\b', "BigQuery"),
        (r'\bdatabricks\b', "Databricks"),
        (r'\bspark\b', "Spark"),
        (r'\bhadoop\b', "Hadoop"),
        (r'\bairflow\b', "Airflow"),
        (r'\bkafka\b', "Kafka"),
        
        # ═══════════════════════════════════════════════════════════════════════
        # GENERAL PROGRAMMING
        # ═══════════════════════════════════════════════════════════════════════
        (r'\bpython\b', "Python"),
        (r'\bjava\b(?!script)', "Java"),
        (r'\bjavascript\b', "JavaScript"),
        (r'\btypescript\b', "TypeScript"),
        (r'\bjs\b', "JavaScript"),
        (r'\bts\b', "TypeScript"),
        (r'\bgolang\b|\bgo\b(?:\s+developer|\s+engineer)?', "Go"),
        (r'\brust\b', "Rust"),
        (r'\bc\+\+\b', "C++"),
        (r'\bc#\b|\.net\b|dotnet\b', "C#/.NET"),
        (r'\bphp\b', "PHP"),
        (r'\bruby\b', "Ruby"),
        (r'\bscala\b', "Scala"),
        (r'\bkotlin\b', "Kotlin"),
        (r'\bswift\b', "Swift"),
        
        # Frontend
        (r'\breact\b(?:\s*native)?', "React"),
        (r'\bangular\b', "Angular"),
        (r'\bvue\b|vuejs', "Vue.js"),
        (r'\bnodejs\b|node\.js\b|\bnode\b', "Node.js"),
        (r'\bfrontend\b|front-end\b|front\s*end\b', "Frontend"),
        
        # Backend & NoSQL Databases
        (r'\bbackend\b|back-end\b|back\s*end\b', "Backend"),
        (r'\bfullstack\b|full-stack\b|full\s*stack\b', "Full Stack"),
        (r'\bsql\b', "SQL"),
        (r'\bmongodb\b|mongo\b', "MongoDB"),
        (r'\bredis\b', "Redis"),
        
        # Cloud & DevOps
        (r'\baws\b|amazon\s*web\s*services', "AWS"),
        (r'\bazure\b', "Azure"),
        (r'\bgcp\b|google\s*cloud', "GCP"),
        (r'\bkubernetes\b|k8s\b', "Kubernetes"),
        (r'\bdocker\b', "Docker"),
        (r'\bterraform\b', "Terraform"),
        (r'\bdevops\b', "DevOps"),
        (r'\bsre\b|site\s*reliability', "SRE"),
        (r'\bci/cd\b|cicd\b', "CI/CD"),
        (r'\bjenkins\b', "Jenkins"),
        (r'\blinux\b', "Linux"),
        
        # Data & AI
        (r'\bmachine\s*learning\b|ml\s*engineer', "Machine Learning"),
        (r'\bdata\s*engineer', "Data Engineering"),
        (r'\bdata\s*scien', "Data Science"),
        (r'\bai\b|artificial\s*intelligence', "AI"),
        (r'\bdeep\s*learning\b', "Deep Learning"),
        (r'\btensorflow\b', "TensorFlow"),
        (r'\bpytorch\b', "PyTorch"),
        
        # Mobile
        (r'\bandroid\b', "Android"),
        (r'\bios\b', "iOS"),
        (r'\bflutter\b', "Flutter"),
        (r'\bmobile\b', "Mobile"),
        
        # Other
        (r'\bapi\b', "API"),
        (r'\bgraphql\b', "GraphQL"),
        (r'\bmicroservices\b', "Microservices"),
        (r'\bagile\b', "Agile"),
        (r'\bscrum\b', "Scrum"),
        (r'\bqa\b|quality\s*assurance|test\s*engineer', "QA"),
        (r'\bsecurity\b|cybersecurity\b', "Security"),
    ]
    
    found_skills = []
    for pattern, display_name in skill_patterns:
        if re.search(pattern, t):
            if display_name not in found_skills:
                found_skills.append(display_name)
    
    # Return unique skills, max 5
    return ", ".join(found_skills[:5])

def parse_date(value) -> datetime:
    """Parse various date formats into datetime."""
    if value is None:
        return None
    if isinstance(value, (int, float)):
        try:
            if value > 10**12:
                return datetime.fromtimestamp(value / 1000, tz=timezone.utc)
            elif value > 10**9:
                return datetime.fromtimestamp(value, tz=timezone.utc)
        except:
            pass
        return None
    s = str(value).strip()
    for fmt in ("%Y-%m-%dT%H:%M:%S.%fZ", "%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%d"):
        try:
            return datetime.strptime(s, fmt).replace(tzinfo=timezone.utc)
        except:
            pass
    return None

def is_within_days(dt: datetime, days: int) -> bool:
    """Check if datetime is within the last N days."""
    if not isinstance(dt, datetime):
        return False
    cutoff = datetime.now(timezone.utc) - timedelta(days=days)
    return dt >= cutoff

def normalize_url(url: str) -> str:
    """Normalize URL for deduplication."""
    if not url:
        return ""
    try:
        parsed = urlparse(url.strip().lower())
        remove_params = {'ckey', 'ckeyword', 'sid', 'utm_source', 'utm_medium', 
                         'utm_campaign', 'ref', 'source', 'click_id'}
        params = parse_qs(parsed.query, keep_blank_values=False)
        filtered_params = {k: v for k, v in params.items() if k.lower() not in remove_params}
        clean_query = urlencode(filtered_params, doseq=True) if filtered_params else ""
        return urlunparse((parsed.scheme, parsed.netloc, parsed.path.rstrip('/'), 
                          parsed.params, clean_query, ""))
    except:
        return url.strip().lower()

def create_dedup_key(job: Dict) -> str:
    """Create a deduplication key for a job."""
    source = job.get("source", "").lower()
    if source == "jooble":
        return f"jooble:{norm(job.get('company',''))}|{norm(job.get('title',''))}|{norm(job.get('location',''))}"
    url = normalize_url(job.get("url", ""))
    if url:
        return f"url:{url}"
    return f"meta:{norm(job.get('company',''))}|{norm(job.get('title',''))}|{norm(job.get('location',''))}"

def http_get(url: str, params: dict = None) -> Any:
    """Safe HTTP GET returning JSON or None."""
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
        if r.status_code != 200:
            return None
        return r.json()
    except:
        return None

def http_post_json(url: str, payload: dict) -> Any:
    """Safe HTTP POST with JSON body."""
    try:
        headers = {"Content-Type": "application/json", "User-Agent": "Mozilla/5.0"}
        r = requests.post(url, headers=headers, data=json.dumps(payload), timeout=TIMEOUT)
        if r.status_code != 200:
            return None
        return r.json()
    except:
        return None

print("✅ Helper functions defined!")

✅ Helper functions defined!


In [12]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              JOB FETCHERS
# ════════════════════════════════════════════════════════════════════════════════════════

def fetch_greenhouse(board: str) -> List[Dict]:
    """Fetch jobs from Greenhouse Job Board API."""
    url = f"https://boards-api.greenhouse.io/v1/boards/{board}/jobs"
    data = http_get(url)
    jobs = []
    if not data or "jobs" not in data:
        return jobs
    for j in data.get("jobs", []):
        loc = ""
        loc_data = j.get("location")
        if isinstance(loc_data, dict):
            loc = loc_data.get("name", "")
        elif isinstance(loc_data, str):
            loc = loc_data
        jobs.append({
            "source": "Greenhouse",
            "company": board.title().replace("-", " "),
            "title": j.get("title", ""),
            "location": loc,
            "department": "",
            "url": j.get("absolute_url", ""),
            "date": parse_date(j.get("updated_at") or j.get("created_at")),
        })
    return jobs

def fetch_lever(site: str) -> List[Dict]:
    """Fetch jobs from Lever Postings API."""
    url = f"https://api.lever.co/v0/postings/{site}"
    data = http_get(url, params={"mode": "json"})
    jobs = []
    if not isinstance(data, list):
        return jobs
    for j in data:
        cats = j.get("categories") or {}
        jobs.append({
            "source": "Lever",
            "company": site.title().replace("-", " "),
            "title": j.get("text", "") or j.get("title", ""),
            "location": cats.get("location", "") or j.get("workplaceType", ""),
            "department": cats.get("team", "") or cats.get("department", ""),
            "url": j.get("hostedUrl", "") or j.get("applyUrl", ""),
            "date": parse_date(j.get("createdAt")),
        })
    return jobs

def fetch_workable(slug: str) -> List[Dict]:
    """Fetch jobs from Workable public widget API."""
    url = f"https://apply.workable.com/api/v1/widget/accounts/{slug}"
    data = http_get(url)
    jobs = []
    if not data:
        return jobs
    company_name = slug.title().replace("-", " ")
    account = data.get("account")
    if isinstance(account, dict):
        company_name = account.get("name", company_name)
    job_list = data.get("jobs") or data.get("results") or []
    if not isinstance(job_list, list):
        return jobs
    for j in job_list:
        loc = j.get("location", "")
        if isinstance(loc, dict):
            loc = loc.get("city", "") or loc.get("name", "")
        jobs.append({
            "source": "Workable",
            "company": company_name,
            "title": j.get("title", "") or j.get("full_title", ""),
            "location": loc,
            "department": j.get("department", "") or j.get("function", ""),
            "url": j.get("url", "") or j.get("shortlink", "") or j.get("application_url", ""),
            "date": parse_date(j.get("published") or j.get("created_at")),
        })
    return jobs

def fetch_jooble(keyword: str, location: str = "Egypt", max_pages: int = 2) -> List[Dict]:
    """Fetch jobs from Jooble API."""
    if not JOOBLE_API_KEY or not JOOBLE_ENABLED:
        return []
    url = f"https://jooble.org/api/{JOOBLE_API_KEY}"
    jobs = []
    for page in range(1, max_pages + 1):
        payload = {"keywords": keyword, "location": location, "page": page}
        data = http_post_json(url, payload)
        if not data:
            break
        job_list = data.get("jobs") or []
        if not job_list:
            break
        for j in job_list:
            dt = parse_date(j.get("updated"))
            if dt and not is_within_days(dt, JOOBLE_DAYS_BACK):
                continue
            company = (j.get("company") or "").strip()
            company = re.sub(r'\s*[-–|•]\s*.*$', '', company)[:50]
            title = re.sub(r'&\w+;', '', (j.get("title") or "").strip())
            jobs.append({
                "source": "Jooble",
                "company": company or "Unknown",
                "title": title,
                "location": j.get("location", ""),
                "department": "",
                "url": j.get("link", ""),
                "date": dt,
            })
        if len(job_list) < 20:
            break
    return jobs

print("✅ Fetcher functions defined!")

✅ Fetcher functions defined!


In [13]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              WUZZUF SCRAPER (Egypt's Largest Job Board)
# ════════════════════════════════════════════════════════════════════════════════════════

def parse_wuzzuf_relative_time(time_str: str) -> datetime:
    """
    Parse Wuzzuf's relative time strings into datetime.
    Examples: '14 minutes ago', '1 hour ago', '2 days ago', 'yesterday'
    """
    if not time_str:
        return None
    
    t = time_str.lower().strip()
    now = datetime.now(timezone.utc)
    
    # Handle "just now" or "moments ago"
    if "just now" in t or "moment" in t:
        return now
    
    # Handle "yesterday"
    if "yesterday" in t:
        return now - timedelta(days=1)
    
    # Parse patterns like "X minutes/hours/days ago"
    match = re.search(r'(\d+)\s*(minute|hour|day|week|month)s?\s*ago', t)
    if match:
        value = int(match.group(1))
        unit = match.group(2)
        
        if unit == "minute":
            return now - timedelta(minutes=value)
        elif unit == "hour":
            return now - timedelta(hours=value)
        elif unit == "day":
            return now - timedelta(days=value)
        elif unit == "week":
            return now - timedelta(weeks=value)
        elif unit == "month":
            return now - timedelta(days=value * 30)  # Approximate
    
    return None

def is_within_n_days(dt: datetime, days: int) -> bool:
    """Check if datetime is within the last N days."""
    if not dt:
        return False
    now = datetime.now(timezone.utc)
    return (now - dt) <= timedelta(days=days)

def fetch_wuzzuf_page(page: int = 0, query: str = "") -> List[Dict]:
    """
    Fetch a single page of jobs from Wuzzuf.
    
    Args:
        page: Page number (0-indexed)
        query: Search query (optional)
    
    Returns:
        List of job dictionaries
    """
    # Build URL - Wuzzuf uses 'start' parameter for pagination
    base_url = "https://wuzzuf.net/search/jobs/"
    params = f"?q={query}&start={page}"
    url = base_url + params
    
    try:
        resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if resp.status_code != 200:
            return []
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        jobs = []
        
        # Find job cards using the css-pkv5jc class (job card container)
        job_cards = soup.select('div.css-pkv5jc')
        
        for card in job_cards:
            # Find the job title link
            title_link = card.find('a', href=re.compile(r'/jobs/p/'))
            if not title_link:
                continue
            
            title = title_link.get_text(strip=True)
            href = title_link.get('href', '')
            full_url = href if href.startswith('http') else f"https://wuzzuf.net{href}"
            
            # Find company name - look for link to /jobs/careers/ with text content
            # (there may be multiple links, first is usually logo, second has company name)
            company = "Confidential"
            company_links = card.find_all('a', href=re.compile(r'/jobs/careers/'))
            for company_link in company_links:
                text = company_link.get_text(strip=True)
                if text:  # Get the first link that has actual text
                    company = text.replace(' -', '').strip()
                    break
            
            # Extract location from job URL or card text
            location = "Egypt"
            url_parts = href.split('-')
            if len(url_parts) >= 2:
                last_parts = '-'.join(url_parts[-3:]).lower()
                if 'cairo' in last_parts:
                    location = "Cairo, Egypt"
                elif 'giza' in last_parts:
                    location = "Giza, Egypt"
                elif 'alexandria' in last_parts:
                    location = "Alexandria, Egypt"
                elif 'egypt' in last_parts:
                    location = "Egypt"
            
            # Find posted time - look for text containing "ago"
            posted_time = None
            for elem in card.find_all(['span', 'div']):
                text = elem.get_text(strip=True).lower()
                if ('ago' in text or 'yesterday' in text) and len(text) < 30:
                    posted_time = parse_wuzzuf_relative_time(text)
                    if posted_time:
                        break
            
            jobs.append({
                "source": "Wuzzuf",
                "company": company,
                "title": title,
                "location": location,
                "department": "",
                "url": full_url,
                "date": posted_time,
            })
        
        return jobs
        
    except Exception as e:
        print(f"⚠️ Wuzzuf page {page} error: {e}")
        return []

def fetch_wuzzuf_jobs(max_pages: int = 10, query: str = "", days_back: int = 3) -> List[Dict]:
    """
    Fetch jobs from Wuzzuf with pagination.
    
    Args:
        max_pages: Maximum number of pages to fetch (15 jobs per page)
        query: Search query (e.g., 'developer', 'software engineer')
        days_back: Only return jobs posted in the last N days
    
    Returns:
        List of job dictionaries
    """
    all_jobs = []
    seen_urls = set()
    
    print(f"🔍 Fetching Wuzzuf jobs (max {max_pages} pages, last {days_back} days)...")
    
    for page in range(max_pages):
        page_jobs = fetch_wuzzuf_page(page, query)
        
        if not page_jobs:
            print(f"   Page {page + 1}: No more jobs found, stopping.")
            break
        
        # Track new jobs (avoid duplicates)
        new_count = 0
        old_count = 0
        
        for job in page_jobs:
            url = job.get("url", "").strip().lower()
            if url and url not in seen_urls:
                seen_urls.add(url)
                
                # Check if within days_back
                if is_within_n_days(job.get("date"), days_back):
                    all_jobs.append(job)
                    new_count += 1
                else:
                    old_count += 1
        
        print(f"   Page {page + 1}: Found {new_count} recent jobs" + 
              (f", skipped {old_count} older jobs" if old_count > 0 else ""))
        
        # If all jobs on this page are older, stop pagination
        if new_count == 0 and old_count > 0:
            print(f"   Stopping: All jobs on page {page + 1} are older than {days_back} days.")
            break
    
    print(f"✅ Wuzzuf: Found {len(all_jobs)} jobs from the last {days_back} days")
    return all_jobs

print("✅ Wuzzuf scraper functions defined!")

✅ Wuzzuf scraper functions defined!


In [14]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              REMOTE JOB SOURCES (For Egyptians seeking remote work)
# ════════════════════════════════════════════════════════════════════════════════════════

def fetch_remoteok_jobs(tags: List[str] = None) -> List[Dict]:
    """
    Fetch jobs from RemoteOK API - one of the largest remote job boards.
    All jobs are remote, making them ideal for Egyptians.
    
    API: https://remoteok.com/api
    """
    if not REMOTEOK_ENABLED:
        return []
    
    print("🌐 Fetching RemoteOK jobs (global remote positions)...")
    
    url = "https://remoteok.com/api"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json"
    }
    
    try:
        resp = requests.get(url, headers=headers, timeout=TIMEOUT)
        if resp.status_code != 200:
            print(f"   ⚠️ RemoteOK returned status {resp.status_code}")
            return []
        
        data = resp.json()
        jobs = []
        
        # First item is metadata, skip it
        job_list = data[1:] if isinstance(data, list) and len(data) > 1 else []
        
        for j in job_list:
            # Filter by tags if specified
            job_tags = [t.lower() for t in (j.get("tags") or [])]
            if tags:
                if not any(tag.lower() in job_tags or tag.lower() in j.get("position", "").lower() 
                          for tag in tags):
                    continue
            
            # Parse date
            date_str = j.get("date")
            dt = None
            if date_str:
                try:
                    dt = datetime.fromisoformat(date_str.replace("Z", "+00:00"))
                except:
                    pass
            
            # Build location string (always remote, but may specify region)
            location = j.get("location") or "Remote (Worldwide)"
            if "remote" not in location.lower():
                location = f"Remote - {location}"
            
            jobs.append({
                "source": "RemoteOK",
                "company": j.get("company") or "Unknown",
                "title": j.get("position") or j.get("title") or "",
                "location": location,
                "department": ", ".join(job_tags[:3]) if job_tags else "",
                "url": j.get("url") or f"https://remoteok.com/remote-jobs/{j.get('id', '')}",
                "date": dt,
            })
        
        print(f"   ✅ RemoteOK: Found {len(jobs)} remote tech jobs")
        return jobs
        
    except Exception as e:
        print(f"   ⚠️ RemoteOK error: {e}")
        return []


def fetch_remotive_jobs(categories: List[str] = None) -> List[Dict]:
    """
    Fetch jobs from Remotive API - curated remote job board.
    
    API: https://remotive.com/api/remote-jobs
    Categories: software-dev, data, devops-sysadmin, product, qa, design, etc.
    """
    if not REMOTIVE_ENABLED:
        return []
    
    print("🌐 Fetching Remotive jobs (curated remote positions)...")
    
    all_jobs = []
    categories_to_fetch = categories or ["software-dev"]
    
    for category in categories_to_fetch:
        url = f"https://remotive.com/api/remote-jobs?category={category}"
        
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if resp.status_code != 200:
                continue
            
            data = resp.json()
            job_list = data.get("jobs") or []
            
            for j in job_list:
                # Parse date
                pub_date = j.get("publication_date")
                dt = None
                if pub_date:
                    try:
                        dt = datetime.fromisoformat(pub_date.replace("Z", "+00:00"))
                    except:
                        try:
                            dt = datetime.strptime(pub_date, "%Y-%m-%dT%H:%M:%S")
                            dt = dt.replace(tzinfo=timezone.utc)
                        except:
                            pass
                
                # Location - Remotive jobs are all remote
                location = j.get("candidate_required_location") or "Remote (Worldwide)"
                if location and "remote" not in location.lower():
                    location = f"Remote - {location}"
                
                all_jobs.append({
                    "source": "Remotive",
                    "company": j.get("company_name") or "Unknown",
                    "title": j.get("title") or "",
                    "location": location,
                    "department": j.get("category") or category,
                    "url": j.get("url") or "",
                    "date": dt,
                })
                
        except Exception as e:
            print(f"   ⚠️ Remotive {category} error: {e}")
            continue
    
    print(f"   ✅ Remotive: Found {len(all_jobs)} remote tech jobs")
    return all_jobs


def fetch_himalayas_jobs() -> List[Dict]:
    """
    Fetch jobs from Himalayas - remote job board with many tech positions.
    
    API: https://himalayas.app/jobs/api
    """
    print("🏔️ Fetching Himalayas jobs (remote tech positions)...")
    
    url = "https://himalayas.app/jobs/api"
    params = {"limit": 100}
    
    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=TIMEOUT)
        if resp.status_code != 200:
            print(f"   ⚠️ Himalayas returned status {resp.status_code}")
            return []
        
        data = resp.json()
        job_list = data.get("jobs") or []
        jobs = []
        
        for j in job_list:
            # Parse date
            pub_date = j.get("pubDate") or j.get("published_at")
            dt = None
            if pub_date:
                try:
                    dt = datetime.fromisoformat(pub_date.replace("Z", "+00:00"))
                except:
                    pass
            
            # Categories/tags for tech filtering
            categories = j.get("categories") or []
            if isinstance(categories, list):
                categories = [c.get("name", c) if isinstance(c, dict) else c for c in categories]
            
            # Location
            location = j.get("location") or "Remote (Worldwide)"
            if "remote" not in location.lower():
                location = f"Remote - {location}"
            
            jobs.append({
                "source": "Himalayas",
                "company": j.get("companyName") or j.get("company", {}).get("name", "Unknown"),
                "title": j.get("title") or "",
                "location": location,
                "department": ", ".join(categories[:3]) if categories else "",
                "url": j.get("applicationUrl") or j.get("url") or "",
                "date": dt,
            })
        
        print(f"   ✅ Himalayas: Found {len(jobs)} remote tech jobs")
        return jobs
        
    except Exception as e:
        print(f"   ⚠️ Himalayas error: {e}")
        return []


def fetch_jobicy_jobs() -> List[Dict]:
    """
    Fetch jobs from Jobicy RSS/API - another remote job board.
    
    API: https://jobicy.com/api/v2/remote-jobs
    """
    print("💼 Fetching Jobicy jobs (remote positions)...")
    
    url = "https://jobicy.com/api/v2/remote-jobs"
    params = {"count": 50, "industry": "dev"}
    
    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=TIMEOUT)
        if resp.status_code != 200:
            print(f"   ⚠️ Jobicy returned status {resp.status_code}")
            return []
        
        data = resp.json()
        job_list = data.get("jobs") or []
        jobs = []
        
        for j in job_list:
            # Parse date
            pub_date = j.get("pubDate")
            dt = None
            if pub_date:
                try:
                    dt = datetime.strptime(pub_date, "%Y-%m-%d %H:%M:%S")
                    dt = dt.replace(tzinfo=timezone.utc)
                except:
                    pass
            
            # Location
            geo = j.get("jobGeo") or "Worldwide"
            location = f"Remote - {geo}"
            
            jobs.append({
                "source": "Jobicy",
                "company": j.get("companyName") or "Unknown",
                "title": j.get("jobTitle") or "",
                "location": location,
                "department": j.get("jobIndustry") or "",
                "url": j.get("url") or "",
                "date": dt,
            })
        
        print(f"   ✅ Jobicy: Found {len(jobs)} remote tech jobs")
        return jobs
        
    except Exception as e:
        print(f"   ⚠️ Jobicy error: {e}")
        return []


print("✅ Remote job source functions defined!")
print("   Sources: RemoteOK, Remotive, Himalayas, Jobicy")

✅ Remote job source functions defined!
   Sources: RemoteOK, Remotive, Himalayas, Jobicy


In [15]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                    ADDITIONAL EGYPTIAN JOB SOURCES (Scrapers)
# ════════════════════════════════════════════════════════════════════════════════════════

def fetch_forasna_page(page: int = 1, query: str = "developer") -> List[Dict]:
    """Fetch jobs from Forasna.com - Egyptian job board."""
    url = f"https://www.forasna.com/jobs"
    params = {"q": query, "page": page}
    
    try:
        resp = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
        if resp.status_code != 200:
            return []
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        jobs = []
        
        # Find job cards
        job_cards = soup.select('div.job-card, div.job-listing, article.job, div.card')
        
        for card in job_cards:
            title_elem = card.select_one('h2 a, h3 a, .job-title a, a.job-link, .card-title a')
            if not title_elem:
                continue
            
            title = title_elem.get_text(strip=True)
            href = title_elem.get('href', '')
            full_url = href if href.startswith('http') else f"https://www.forasna.com{href}"
            
            company_elem = card.select_one('.company-name, .employer, span.company, .card-subtitle')
            company = company_elem.get_text(strip=True) if company_elem else "Unknown"
            
            location_elem = card.select_one('.location, .job-location, span.location')
            location = location_elem.get_text(strip=True) if location_elem else "Egypt"
            if location and "egypt" not in location.lower():
                location = f"{location}, Egypt"
            
            date_elem = card.select_one('.date, .posted-date, time, .text-muted')
            posted_time = parse_wuzzuf_relative_time(date_elem.get_text(strip=True)) if date_elem else None
            
            jobs.append({
                "source": "Forasna",
                "company": company,
                "title": title,
                "location": location,
                "department": "",
                "url": full_url,
                "date": posted_time,
            })
        
        return jobs
    except Exception as e:
        return []


def fetch_forasna_jobs(max_pages: int = 5, days_back: int = 14) -> List[Dict]:
    """Fetch jobs from Forasna with multiple search queries."""
    if not FORASNA_ENABLED:
        return []
    
    print("🇪🇬 Fetching Forasna jobs...")
    
    queries = ["software developer", "software engineer", "data engineer", "devops", 
               "backend developer", "frontend developer", "mobile developer", "qa engineer",
               "database developer", "oracle developer", "sql developer", "python", "java"]
    
    all_jobs = []
    seen_urls = set()
    
    for query in queries:
        for page in range(1, max_pages + 1):
            page_jobs = fetch_forasna_page(page, query)
            if not page_jobs:
                break
            
            for job in page_jobs:
                url = job.get("url", "").strip().lower()
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    if is_within_n_days(job.get("date"), days_back) or job.get("date") is None:
                        all_jobs.append(job)
    
    print(f"   ✅ Forasna: Found {len(all_jobs)} jobs")
    return all_jobs


def fetch_indeed_egypt_page(query: str = "software developer", start: int = 0) -> List[Dict]:
    """Fetch jobs from Indeed Egypt."""
    url = "https://eg.indeed.com/jobs"
    params = {"q": query, "l": "Egypt", "start": start}
    
    try:
        resp = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
        if resp.status_code != 200:
            return []
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        jobs = []
        
        # Find job cards
        job_cards = soup.select('div.job_seen_beacon, div.jobsearch-ResultsList > div, div.slider_container, div.result')
        
        for card in job_cards:
            title_elem = card.select_one('h2.jobTitle a, a.jcs-JobTitle, span[title], h2 a')
            if not title_elem:
                continue
            
            title = title_elem.get_text(strip=True) or title_elem.get('title', '')
            href = title_elem.get('href', '')
            if href and not href.startswith('http'):
                href = f"https://eg.indeed.com{href}"
            
            company_elem = card.select_one('span.companyName, span[data-testid="company-name"], .company')
            company = company_elem.get_text(strip=True) if company_elem else "Unknown"
            
            location_elem = card.select_one('div.companyLocation, span[data-testid="text-location"], .location')
            location = location_elem.get_text(strip=True) if location_elem else "Egypt"
            
            date_elem = card.select_one('span.date, span[data-testid="myJobsStateDate"]')
            posted_time = parse_wuzzuf_relative_time(date_elem.get_text(strip=True)) if date_elem else None
            
            if title and href:
                jobs.append({
                    "source": "Indeed Egypt",
                    "company": company,
                    "title": title,
                    "location": location if "egypt" in location.lower() else f"{location}, Egypt",
                    "department": "",
                    "url": href,
                    "date": posted_time,
                })
        
        return jobs
    except Exception as e:
        return []


def fetch_indeed_egypt_jobs(max_pages: int = 5, days_back: int = 14) -> List[Dict]:
    """Fetch jobs from Indeed Egypt with pagination."""
    if not INDEED_EGYPT_ENABLED:
        return []
    
    print("🇪🇬 Fetching Indeed Egypt jobs...")
    
    queries = ["software developer", "software engineer", "data engineer", "devops engineer",
               "oracle developer", "database developer", "python developer", "java developer",
               "frontend developer", "backend developer", "mobile developer"]
    
    all_jobs = []
    seen_urls = set()
    
    for query in queries:
        for page in range(max_pages):
            start = page * 10
            page_jobs = fetch_indeed_egypt_page(query, start)
            if not page_jobs:
                break
            
            for job in page_jobs:
                url = job.get("url", "").strip().lower()
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    all_jobs.append(job)
    
    print(f"   ✅ Indeed Egypt: Found {len(all_jobs)} jobs")
    return all_jobs


def fetch_bayt_page(query: str = "software-developer", page: int = 1) -> List[Dict]:
    """Fetch jobs from Bayt.com - Major MENA job board."""
    url = f"https://www.bayt.com/en/egypt/jobs/{query}-jobs/"
    if page > 1:
        url += f"?page={page}"
    
    try:
        resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if resp.status_code != 200:
            return []
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        jobs = []
        
        job_cards = soup.select('li[data-js-job], div.job-card, li.has-pointer-d, div[data-job-id]')
        
        for card in job_cards:
            title_elem = card.select_one('h2 a, a.jb-title, .job-title a, a[data-js-job-title]')
            if not title_elem:
                continue
            
            title = title_elem.get_text(strip=True)
            href = title_elem.get('href', '')
            full_url = href if href.startswith('http') else f"https://www.bayt.com{href}"
            
            company_elem = card.select_one('.jb-company, .company-name, b[data-js-company-name], .t-default')
            company = company_elem.get_text(strip=True) if company_elem else "Unknown"
            
            location_elem = card.select_one('.jb-location, .job-location, span[data-js-location]')
            location = location_elem.get_text(strip=True) if location_elem else "Egypt"
            
            date_elem = card.select_one('.jb-date, .posted-date, time')
            posted_time = parse_wuzzuf_relative_time(date_elem.get_text(strip=True)) if date_elem else None
            
            jobs.append({
                "source": "Bayt.com",
                "company": company,
                "title": title,
                "location": location,
                "department": "",
                "url": full_url,
                "date": posted_time,
            })
        
        return jobs
    except Exception as e:
        return []


def fetch_bayt_jobs(max_pages: int = 5, days_back: int = 14) -> List[Dict]:
    """Fetch jobs from Bayt.com with multiple search queries."""
    if not BAYT_ENABLED:
        return []
    
    print("🌍 Fetching Bayt.com jobs (Egypt)...")
    
    queries = ["software-developer", "software-engineer", "data-engineer", "devops",
               "backend-developer", "frontend-developer", "database-developer", "python", "java"]
    
    all_jobs = []
    seen_urls = set()
    
    for query in queries:
        for page in range(1, max_pages + 1):
            page_jobs = fetch_bayt_page(query, page)
            if not page_jobs:
                break
            
            for job in page_jobs:
                url = job.get("url", "").strip().lower()
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    if is_within_n_days(job.get("date"), days_back) or job.get("date") is None:
                        all_jobs.append(job)
    
    print(f"   ✅ Bayt.com: Found {len(all_jobs)} Egypt jobs")
    return all_jobs


def fetch_linkedin_egypt_jobs() -> List[Dict]:
    """Fetch jobs from LinkedIn public guest search (Egypt)."""
    if not LINKEDIN_ENABLED:
        return []
    
    print("💼 Fetching LinkedIn jobs (Egypt)...")
    
    all_jobs = []
    seen_urls = set()
    
    base_url = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
    
    for keyword in LINKEDIN_KEYWORDS:
        for start in range(0, LINKEDIN_MAX_PAGES * 25, 25):
            params = {
                "keywords": keyword,
                "location": "Egypt",
                "geoId": "106155005",  # Egypt geoId
                "start": start,
                "f_TPR": "r604800",  # Last week
            }
            
            try:
                resp = requests.get(base_url, params=params, headers=HEADERS, timeout=TIMEOUT)
                if resp.status_code != 200:
                    continue
                
                soup = BeautifulSoup(resp.text, 'html.parser')
                job_cards = soup.select('li, div.base-card, div.job-search-card')
                
                if not job_cards:
                    break
                
                for card in job_cards:
                    title_elem = card.select_one('h3.base-search-card__title, a.base-card__full-link, h3')
                    if not title_elem:
                        continue
                    
                    title = title_elem.get_text(strip=True)
                    
                    link_elem = card.select_one('a.base-card__full-link, a[href*="/jobs/view/"]')
                    href = link_elem.get('href', '') if link_elem else ""
                    
                    if not href or href in seen_urls:
                        continue
                    seen_urls.add(href)
                    
                    company_elem = card.select_one('h4.base-search-card__subtitle, a.hidden-nested-link')
                    company = company_elem.get_text(strip=True) if company_elem else "Unknown"
                    
                    location_elem = card.select_one('span.job-search-card__location')
                    location = location_elem.get_text(strip=True) if location_elem else "Egypt"
                    
                    date_elem = card.select_one('time')
                    posted_time = None
                    if date_elem:
                        datetime_attr = date_elem.get('datetime', '')
                        if datetime_attr:
                            try:
                                posted_time = datetime.fromisoformat(datetime_attr.replace("Z", "+00:00"))
                            except:
                                pass
                    
                    all_jobs.append({
                        "source": "LinkedIn",
                        "company": company,
                        "title": title,
                        "location": location,
                        "department": "",
                        "url": href,
                        "date": posted_time,
                    })
                    
            except Exception as e:
                continue
    
    print(f"   ✅ LinkedIn: Found {len(all_jobs)} Egypt jobs")
    return all_jobs


def fetch_jobzella_jobs() -> List[Dict]:
    """Fetch jobs from Jobzella - Egyptian startup-focused job board."""
    if not JOBZELLA_ENABLED:
        return []
    
    print("🇪🇬 Fetching Jobzella jobs...")
    
    all_jobs = []
    seen_urls = set()
    
    queries = ["software", "developer", "engineer", "data", "devops", "backend", "frontend"]
    
    for query in queries:
        url = f"https://www.jobzella.com/jobs/search"
        params = {"q": query, "location": "egypt"}
        
        try:
            resp = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
            if resp.status_code != 200:
                continue
            
            soup = BeautifulSoup(resp.text, 'html.parser')
            job_cards = soup.select('div.job-card, article.job, div.job-item, li.job-listing, div.card')
            
            for card in job_cards:
                title_elem = card.select_one('h2 a, h3 a, a.job-title, .job-title a, .card-title a')
                if not title_elem:
                    continue
                
                title = title_elem.get_text(strip=True)
                href = title_elem.get('href', '')
                
                if not href.startswith('http'):
                    href = f"https://www.jobzella.com{href}"
                
                if href in seen_urls:
                    continue
                seen_urls.add(href)
                
                company_elem = card.select_one('.company-name, .employer-name, a.company, .card-subtitle')
                company = company_elem.get_text(strip=True) if company_elem else "Unknown"
                
                location_elem = card.select_one('.location, .job-location')
                location = location_elem.get_text(strip=True) if location_elem else "Egypt"
                
                date_elem = card.select_one('.date, .posted-date, time, .job-date')
                posted_time = parse_wuzzuf_relative_time(date_elem.get_text(strip=True)) if date_elem else None
                
                all_jobs.append({
                    "source": "Jobzella",
                    "company": company,
                    "title": title,
                    "location": location if "egypt" in location.lower() else f"{location}, Egypt",
                    "department": "",
                    "url": href,
                    "date": posted_time,
                })
                
        except Exception as e:
            continue
    
    print(f"   ✅ Jobzella: Found {len(all_jobs)} jobs")
    return all_jobs


def fetch_tanqeeb_egypt_jobs() -> List[Dict]:
    """Fetch jobs from Tanqeeb - Arab region job aggregator with Egypt jobs."""
    if not TANQEEB_ENABLED:
        return []
    
    print("🌍 Fetching Tanqeeb jobs (Egypt)...")
    
    all_jobs = []
    seen_urls = set()
    
    queries = ["software developer", "software engineer", "data engineer", "devops", "backend", "frontend"]
    
    for query in queries:
        url = "https://egypt.tanqeeb.com/en/jobs/search"
        params = {"q": query}
        
        try:
            resp = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
            if resp.status_code != 200:
                continue
            
            soup = BeautifulSoup(resp.text, 'html.parser')
            job_cards = soup.select('div.job-card, div.job-listing, article.job, li.job-item, div.card')
            
            for card in job_cards:
                title_elem = card.select_one('h2 a, h3 a, a.job-title, .title a, .card-title a')
                if not title_elem:
                    continue
                
                title = title_elem.get_text(strip=True)
                href = title_elem.get('href', '')
                
                if not href.startswith('http'):
                    href = f"https://egypt.tanqeeb.com{href}"
                
                if href in seen_urls:
                    continue
                seen_urls.add(href)
                
                company_elem = card.select_one('.company, .employer, span.company-name, .card-subtitle')
                company = company_elem.get_text(strip=True) if company_elem else "Unknown"
                
                location_elem = card.select_one('.location, .job-location')
                location = location_elem.get_text(strip=True) if location_elem else "Egypt"
                
                all_jobs.append({
                    "source": "Tanqeeb",
                    "company": company,
                    "title": title,
                    "location": location,
                    "department": "",
                    "url": href,
                    "date": None,
                })
                
        except Exception as e:
            continue
    
    print(f"   ✅ Tanqeeb: Found {len(all_jobs)} Egypt jobs")
    return all_jobs


print("✅ Additional Egyptian job scrapers defined!")
print("   Sources: Forasna, Indeed Egypt, Bayt.com, LinkedIn, Jobzella, Tanqeeb")

✅ Additional Egyptian job scrapers defined!
   Sources: Forasna, Indeed Egypt, Bayt.com, LinkedIn, Jobzella, Tanqeeb


In [16]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              MAIN AGGREGATOR
# ════════════════════════════════════════════════════════════════════════════════════════

def run_aggregator() -> pd.DataFrame:
    """Main function to fetch, filter, and return Egypt tech jobs as DataFrame."""
    
    print("=" * 80)
    print("🚀 EGYPT TECH JOBS AGGREGATOR - Starting...")
    print("=" * 80)
    
    # Count sources
    ats_count = len(GREENHOUSE_BOARDS) + len(LEVER_SITES) + len(WORKABLE_SLUGS)
    
    # Egyptian sources list
    egypt_sources = []
    if WUZZUF_ENABLED: egypt_sources.append("Wuzzuf")
    if JOOBLE_ENABLED: egypt_sources.append("Jooble")
    if FORASNA_ENABLED: egypt_sources.append("Forasna")
    if INDEED_EGYPT_ENABLED: egypt_sources.append("Indeed Egypt")
    if BAYT_ENABLED: egypt_sources.append("Bayt.com")
    if LINKEDIN_ENABLED: egypt_sources.append("LinkedIn")
    if JOBZELLA_ENABLED: egypt_sources.append("Jobzella")
    if TANQEEB_ENABLED: egypt_sources.append("Tanqeeb")
    
    print(f"📊 ATS Sources: {ats_count} total (Greenhouse={len(GREENHOUSE_BOARDS)}, Lever={len(LEVER_SITES)}, Workable={len(WORKABLE_SLUGS)})")
    print(f"🇪🇬 Egypt Sources: {', '.join(egypt_sources)}")
    print(f"🌍 Remote Sources: RemoteOK, Remotive, Himalayas, Jobicy")
    print()
    print("🔍 Fetching jobs from all sources...")
    
    # 1. Fetch all jobs concurrently from ATS APIs
    all_jobs = []
    errors = 0
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []
        for board in GREENHOUSE_BOARDS:
            futures.append(executor.submit(fetch_greenhouse, board))
        for site in LEVER_SITES:
            futures.append(executor.submit(fetch_lever, site))
        for slug in WORKABLE_SLUGS:
            futures.append(executor.submit(fetch_workable, slug))
        
        for future in as_completed(futures):
            try:
                result = future.result()
                if result:
                    all_jobs.extend(result)
            except:
                errors += 1
    
    print(f"📥 Jobs from ATS boards: {len(all_jobs)} (failed sources: {errors})")
    
    # 2. Fetch from Jooble API (Egypt-focused)
    if JOOBLE_ENABLED and JOOBLE_API_KEY:
        print(f"🔄 Fetching from Jooble API ({len(JOOBLE_SEARCH_KEYWORDS)} keywords)...")
        jooble_count = 0
        for keyword in JOOBLE_SEARCH_KEYWORDS:
            jobs = fetch_jooble(keyword, "Egypt", JOOBLE_MAX_PAGES)
            jooble_count += len(jobs)
            all_jobs.extend(jobs)
        print(f"📥 Jobs from Jooble: {jooble_count}")
    
    # 3. Fetch from Wuzzuf (Egypt's largest job board)
    if WUZZUF_ENABLED:
        print()
        wuzzuf_jobs = fetch_wuzzuf_jobs(max_pages=WUZZUF_MAX_PAGES, days_back=WUZZUF_DAYS_BACK)
        all_jobs.extend(wuzzuf_jobs)
    
    # ══════════════════════════════════════════════════════════════════════════════════
    # 4. Fetch from ADDITIONAL EGYPTIAN JOB SOURCES
    # ══════════════════════════════════════════════════════════════════════════════════
    print()
    print("🇪🇬 Fetching from additional Egyptian job boards...")
    
    # Forasna
    if FORASNA_ENABLED:
        forasna_jobs = fetch_forasna_jobs(max_pages=FORASNA_MAX_PAGES, days_back=FORASNA_DAYS_BACK)
        all_jobs.extend(forasna_jobs)
    
    # Indeed Egypt
    if INDEED_EGYPT_ENABLED:
        indeed_jobs = fetch_indeed_egypt_jobs(max_pages=INDEED_MAX_PAGES, days_back=INDEED_DAYS_BACK)
        all_jobs.extend(indeed_jobs)
    
    # Bayt.com
    if BAYT_ENABLED:
        bayt_jobs = fetch_bayt_jobs(max_pages=BAYT_MAX_PAGES, days_back=BAYT_DAYS_BACK)
        all_jobs.extend(bayt_jobs)
    
    # LinkedIn
    if LINKEDIN_ENABLED:
        linkedin_jobs = fetch_linkedin_egypt_jobs()
        all_jobs.extend(linkedin_jobs)
    
    # Jobzella
    if JOBZELLA_ENABLED:
        jobzella_jobs = fetch_jobzella_jobs()
        all_jobs.extend(jobzella_jobs)
    
    # Tanqeeb
    if TANQEEB_ENABLED:
        tanqeeb_jobs = fetch_tanqeeb_egypt_jobs()
        all_jobs.extend(tanqeeb_jobs)
    
    print(f"📥 Total jobs after Egyptian sources: {len(all_jobs)}")
    
    # ══════════════════════════════════════════════════════════════════════════════════
    # 5. Fetch from Remote Job Boards (ideal for Egyptians seeking remote work)
    # ══════════════════════════════════════════════════════════════════════════════════
    print()
    print("🌍 Fetching from global remote job boards...")
    
    if REMOTEOK_ENABLED:
        remoteok_jobs = fetch_remoteok_jobs(tags=REMOTEOK_TAGS)
        all_jobs.extend(remoteok_jobs)
    
    if REMOTIVE_ENABLED:
        remotive_jobs = fetch_remotive_jobs(categories=REMOTIVE_CATEGORIES)
        all_jobs.extend(remotive_jobs)
    
    himalayas_jobs = fetch_himalayas_jobs()
    all_jobs.extend(himalayas_jobs)
    
    jobicy_jobs = fetch_jobicy_jobs()
    all_jobs.extend(jobicy_jobs)
    
    print(f"📥 Total jobs after all sources: {len(all_jobs)}")
    
    # 6. Filter: Egypt + Remote + Tech
    print()
    print("🔧 Filtering jobs...")
    
    # Egypt source list for filtering
    egypt_source_list = ["Wuzzuf", "Jooble", "Forasna", "Indeed Egypt", "Bayt.com", 
                         "LinkedIn", "Jobzella", "Tanqeeb"]
    remote_source_list = ["RemoteOK", "Remotive", "Himalayas", "Jobicy"]
    
    filtered = []
    for job in all_jobs:
        title = job.get("title", "")
        loc = job.get("location", "")
        dept = job.get("department", "")
        company = job.get("company", "")
        url = job.get("url", "")
        source = job.get("source", "")
        
        if not url:
            continue
        
        is_remote_source = source in remote_source_list
        is_remote_job_loc = "remote" in loc.lower() or "worldwide" in loc.lower()
        is_egypt_source = source in egypt_source_list
        
        if EGYPT_ONLY:
            if not is_egypt_location(loc, company) and not is_remote_source and not is_remote_job_loc and not is_egypt_source:
                continue
        
        if TECH_ONLY and not is_tech_job(title, dept):
            continue
        filtered.append(job)
    
    # 7. Deduplicate
    seen_keys = set()
    unique_jobs = []
    for job in filtered:
        key = create_dedup_key(job)
        if key and key not in seen_keys:
            seen_keys.add(key)
            unique_jobs.append(job)
    
    # 8. Sort: Egypt jobs first, then Remote jobs, then by date
    def sort_key(j):
        loc = j.get("location", "")
        company = j.get("company", "")
        source = j.get("source", "")
        country = detect_country(loc, company)
        dt = j.get("date")
        
        is_remote = "remote" in loc.lower() or source in remote_source_list
        is_egypt_source = source in egypt_source_list
        
        if country == "Egypt" or is_egypt_source:
            priority = 0
        elif is_remote:
            priority = 1
        else:
            priority = 2
        
        if isinstance(dt, datetime):
            date_sort = (0, -dt.timestamp())
        else:
            date_sort = (1, 0)
        
        return (priority, date_sort[0], date_sort[1])
    
    unique_jobs.sort(key=sort_key)
    
    # Count stats
    egypt_count = sum(1 for j in unique_jobs 
                      if detect_country(j.get("location", ""), j.get("company", "")) == "Egypt" 
                      or j.get("source", "") in egypt_source_list)
    remote_count = sum(1 for j in unique_jobs 
                       if "remote" in j.get("location", "").lower() 
                       or j.get("source", "") in remote_source_list)
    
    print()
    print("=" * 80)
    print(f"✅ Total tech jobs found: {len(unique_jobs)}")
    print(f"   🇪🇬 Egypt-based: {egypt_count}")
    print(f"   🌍 Remote (work from Egypt): {remote_count}")
    print("=" * 80)
    
    # 9. Create DataFrame with Source Policy Data
    # Each job now references source_id and inherits policy from JOB_SOURCES registry
    
    def build_job_record(j: Dict) -> Dict:
        """Build a job record with full source policy information."""
        source_name = j.get("source", "")
        policy = get_source_policy(source_name)
        
        # Generate unique job_id
        job_id = hash(f"{j.get('title', '')}{j.get('company', '')}{j.get('url', '')}")
        
        return {
            # ═══════════════════════════════════════════════════════════════
            # JOB CORE DATA
            # ═══════════════════════════════════════════════════════════════
            "Job_ID": job_id,
            "Title": j.get("title", ""),
            "Company": j.get("company", ""),
            "Level": detect_experience_level(j.get("title", "")),
            "Salary": extract_salary(f"{j.get('title', '')} {j.get('location', '')}"),
            "Experience_Years": extract_years_experience(j.get("title", "")),
            "Skills": extract_skills(f"{j.get('title', '')} {j.get('department', '')}"),
            
            # ═══════════════════════════════════════════════════════════════
            # SOURCE POLICY DATA (from JOB_SOURCES registry)
            # ═══════════════════════════════════════════════════════════════
            "Source": source_name,
            "Source_ID": policy.source_id,
            "Source_Type": policy.source_type.value,  # official_api, rss_feed, scraping, manual
            "Allowed_Mode": policy.allowed_mode.value,  # full_display, limited_display, link_only, disabled
            "Attribution_Required": "Yes" if policy.attribution_required else "No",
            "Source_URL": policy.source_url,
            "Rate_Limit_RPM": policy.rate_limit.requests_per_minute,
            "Rate_Limit_Burst": policy.rate_limit.burst,
            "Takedown_Contact": policy.takedown_contact,
            "Terms_URL": policy.terms_url,
            "Source_Notes": policy.notes,
            
            # ═══════════════════════════════════════════════════════════════
            # LOCATION DATA
            # ═══════════════════════════════════════════════════════════════
            "Country": detect_country(j.get("location", ""), j.get("company", "")),
            "City": detect_egypt_city(j.get("location", ""), j.get("company", "")),
            "Work_Type": get_work_type_for_egyptians(j.get("location", ""), j.get("title", ""), j.get("company", "")),
            "Location": j.get("location", ""),
            
            # ═══════════════════════════════════════════════════════════════
            # LINKS & DATES
            # ═══════════════════════════════════════════════════════════════
            "Apply_URL": j.get("url", ""),
            "Date": (j.get("date").strftime("%Y-%m-%d") if isinstance(j.get("date"), datetime) else ""),
        }
    
    # Build DataFrame with all policy data
    df = pd.DataFrame([build_job_record(j) for j in unique_jobs])
    
    return df

# Run the aggregator
print("Starting job aggregation...\n")
import time
start_time = time.time()

df_jobs = run_aggregator()

elapsed = time.time() - start_time
print(f"\n⏱️ Completed in {elapsed:.1f} seconds")
print(f"📋 Final DataFrame shape: {df_jobs.shape}")

Starting job aggregation...

🚀 EGYPT TECH JOBS AGGREGATOR - Starting...
📊 ATS Sources: 107 total (Greenhouse=56, Lever=15, Workable=36)
🇪🇬 Egypt Sources: Wuzzuf, Jooble
🌍 Remote Sources: RemoteOK, Remotive, Himalayas, Jobicy

🔍 Fetching jobs from all sources...
📥 Jobs from ATS boards: 9887 (failed sources: 0)
🔄 Fetching from Jooble API (177 keywords)...
📥 Jobs from Jooble: 7190

🔍 Fetching Wuzzuf jobs (max 15 pages, last 14 days)...
   Page 1: Found 15 recent jobs
   Page 2: Found 15 recent jobs
   Page 3: Found 15 recent jobs
   Page 4: Found 15 recent jobs
   Page 5: Found 15 recent jobs
   Page 6: Found 15 recent jobs
   Page 7: Found 15 recent jobs
   Page 8: Found 15 recent jobs
   Page 9: Found 15 recent jobs
   Page 10: Found 15 recent jobs
   Page 11: Found 15 recent jobs
   Page 12: Found 14 recent jobs
   Page 13: Found 15 recent jobs
   Page 14: Found 15 recent jobs
   Page 15: Found 15 recent jobs
✅ Wuzzuf: Found 224 jobs from the last 14 days

🇪🇬 Fetching from additional E

In [17]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              FINAL OUTPUT - ALL JOBS DataFrame
# ════════════════════════════════════════════════════════════════════════════════════════

# Display the complete jobs DataFrame
print("=" * 80)
print("📊 FINAL OUTPUT: All Jobs with Source Policies & Compliance Data")
print("=" * 80)
print(f"\nTotal Jobs: {len(df_jobs)}")
print(f"Columns ({len(df_jobs.columns)}):")
for i, col in enumerate(df_jobs.columns, 1):
    print(f"   {i:2}. {col}")
print("\n")

# Show summary by country (Egypt first)
print("Jobs by Country:")
print(df_jobs['Country'].value_counts().to_string())
print("\n")

# Show Egypt jobs by city
egypt_jobs = df_jobs[df_jobs['Country'] == 'Egypt']
print(f"🇪🇬 Egypt Jobs by City/Governorate ({len(egypt_jobs)} total):")
print(egypt_jobs['City'].value_counts().to_string())
print("\n")

# Show work type breakdown
print("Jobs by Work Type:")
print(df_jobs['Work_Type'].value_counts().to_string())
print("\n")

# ═══════════════════════════════════════════════════════════════════════════
# SOURCE POLICY BREAKDOWN
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 80)
print("🔒 SOURCE POLICY BREAKDOWN")
print("=" * 80)

# By Source Type (API vs Scraping)
print("\n📡 Jobs by Source Type:")
print(df_jobs['Source_Type'].value_counts().to_string())
print()

# By Allowed Mode (Display Restrictions)
print("🖥️ Jobs by Display Mode:")
mode_counts = df_jobs['Allowed_Mode'].value_counts()
for mode, count in mode_counts.items():
    if mode == 'full_display':
        print(f"   ✅ {mode}: {count} (can show full details)")
    elif mode == 'limited_display':
        print(f"   ⚠️ {mode}: {count} (200 char snippet only)")
    elif mode == 'link_only':
        print(f"   🔗 {mode}: {count} (title/company/link only)")
    elif mode == 'disabled':
        print(f"   ❌ {mode}: {count} (should be excluded)")
print()

# Attribution Required
attr_required = df_jobs[df_jobs['Attribution_Required'] == 'Yes']
print(f"📋 Jobs Requiring Attribution: {len(attr_required)}")
if len(attr_required) > 0:
    print("   Sources requiring attribution:")
    for source in attr_required['Source'].unique():
        count = len(attr_required[attr_required['Source'] == source])
        print(f"   - {source}: {count} jobs")
print()

# By Source with Policy Info
print("📊 Jobs by Source (with Policy):")
for source in df_jobs['Source'].unique():
    source_df = df_jobs[df_jobs['Source'] == source]
    mode = source_df['Allowed_Mode'].iloc[0] if len(source_df) > 0 else 'unknown'
    s_type = source_df['Source_Type'].iloc[0] if len(source_df) > 0 else 'unknown'
    attr = source_df['Attribution_Required'].iloc[0] if len(source_df) > 0 else 'No'
    icon = "✅" if mode == 'full_display' else ("⚠️" if mode == 'limited_display' else ("🔗" if mode == 'link_only' else "❌"))
    print(f"   {icon} {source}: {len(source_df)} jobs | {s_type} | {mode} | attr={attr}")
print("\n")

# ═══════════════════════════════════════════════════════════════════════════
# EXPERIENCE & SALARY
# ═══════════════════════════════════════════════════════════════════════════
print("Jobs by Experience Level:")
print(df_jobs['Level'].value_counts().to_string())
print("\n")

jobs_with_salary = df_jobs[df_jobs['Salary'] != '']
print(f"💰 Jobs with Salary Information: {len(jobs_with_salary)}")
if len(jobs_with_salary) > 0:
    print(jobs_with_salary[['Title', 'Company', 'Salary', 'Country']].head(10).to_string())
print("\n")

jobs_with_exp = df_jobs[df_jobs['Experience_Years'] != '']
print(f"📅 Jobs with Experience Requirements: {len(jobs_with_exp)}")
if len(jobs_with_exp) > 0:
    print(df_jobs['Experience_Years'].value_counts().head(10).to_string())
print("\n")

jobs_with_skills = df_jobs[df_jobs['Skills'] != '']
print(f"🛠️ Jobs with Skills Detected: {len(jobs_with_skills)}")
print("\n")

# Display the full DataFrame (Egypt jobs are sorted first)
print("=" * 80)
print("📋 JOBS TABLE (Egypt jobs first, then other countries):")
print("=" * 80)
display(df_jobs)

# Show tips
print("\n✅ The 'df_jobs' DataFrame is ready for use!")
print("""
   ═══════════════════════════════════════════════════════════════════════════
   COLUMNS REFERENCE:
   ═══════════════════════════════════════════════════════════════════════════
   JOB DATA:       Job_ID, Title, Company, Level, Salary, Experience_Years, Skills
   SOURCE POLICY:  Source, Source_ID, Source_Type, Allowed_Mode, Attribution_Required,
                   Source_URL, Rate_Limit_RPM, Rate_Limit_Burst, Takedown_Contact, 
                   Terms_URL, Source_Notes
   LOCATION:       Country, City, Work_Type, Location
   LINKS:          Apply_URL, Date
   
   ═══════════════════════════════════════════════════════════════════════════
   QUICK FILTERS:
   ═══════════════════════════════════════════════════════════════════════════
   - Egypt only:         df_jobs[df_jobs['Country'] == 'Egypt']
   - Remote jobs:        df_jobs[df_jobs['Work_Type'] == 'Remote']
   - Cairo jobs:         df_jobs[df_jobs['City'] == 'Cairo']
   - Senior jobs:        df_jobs[df_jobs['Level'] == 'Senior']
   
   ═══════════════════════════════════════════════════════════════════════════
   POLICY-BASED FILTERS:
   ═══════════════════════════════════════════════════════════════════════════
   - Full display only:  df_jobs[df_jobs['Allowed_Mode'] == 'full_display']
   - API sources only:   df_jobs[df_jobs['Source_Type'] == 'official_api']
   - Attribution needed: df_jobs[df_jobs['Attribution_Required'] == 'Yes']
   - Exclude disabled:   df_jobs[df_jobs['Allowed_Mode'] != 'disabled']
   
   ═══════════════════════════════════════════════════════════════════════════
   API SERIALIZATION (for your backend):
   ═══════════════════════════════════════════════════════════════════════════
   # Serialize jobs according to their display mode:
   api_response = JobSerializer.get_api_response(df_jobs.to_dict('records'))
   
   # Check rate limit before fetching:
   if RATE_LIMITER.acquire('Wuzzuf'):
       # fetch data
   
   ═══════════════════════════════════════════════════════════════════════════
   EXPORT:
   ═══════════════════════════════════════════════════════════════════════════
   - Export to CSV:      df_jobs.to_csv('egypt_jobs.csv', index=False)
   - Export to JSON:     df_jobs.to_json('egypt_jobs.json', orient='records')
""")

📊 FINAL OUTPUT: All Jobs with Source Policies & Compliance Data

Total Jobs: 1582
Columns (24):
    1. Job_ID
    2. Title
    3. Company
    4. Level
    5. Salary
    6. Experience_Years
    7. Skills
    8. Source
    9. Source_ID
   10. Source_Type
   11. Allowed_Mode
   12. Attribution_Required
   13. Source_URL
   14. Rate_Limit_RPM
   15. Rate_Limit_Burst
   16. Takedown_Contact
   17. Terms_URL
   18. Source_Notes
   19. Country
   20. City
   21. Work_Type
   22. Location
   23. Apply_URL
   24. Date


Jobs by Country:
Country
Remote          507
Egypt           448
Other           307
USA             164
Canada           85
UK               48
UAE              13
Germany           4
Lebanon           2
Jordan            2
Netherlands       1
Saudi Arabia      1


🇪🇬 Egypt Jobs by City/Governorate (448 total):
City
Egypt (General)    348
Cairo               84
Alexandria          10
Giza                 6


Jobs by Work Type:
Work_Type
Remote                 791
On-site       

,Job_ID,Title,Company,Level,Salary,Experience_Years,Skills,Source,Source_ID,Source_Type,...,Rate_Limit_Burst,Takedown_Contact,Terms_URL,Source_Notes,Country,City,Work_Type,Location,Apply_URL,Date
0,-6687376651442586105,Design engineer,Al ayat for printing and packaging,Mid,,,,Wuzzuf,wuzzuf,scraping,...,2,info@wuzzuf.net,https://wuzzuf.net/terms-and-conditions,Egypt's largest job board. Scraping with rate ...,Egypt,Giza,On-site,"Giza, Egypt",https://wuzzuf.net/jobs/p/4vyqmb5lyxgn-design-...,2026-01-23
1,-8139832324940542033,Quality engineer,Al ayat for printing and packaging,Mid,,,,Wuzzuf,wuzzuf,scraping,...,2,info@wuzzuf.net,https://wuzzuf.net/terms-and-conditions,Egypt's largest job board. Scraping with rate ...,Egypt,Giza,On-site,"Giza, Egypt",https://wuzzuf.net/jobs/p/qrpnlnkig5cs-quality...,2026-01-23
2,-5791183466350770923,Maintenance Engineer,Al ayat for printing and packaging,Mid,,,,Wuzzuf,wuzzuf,scraping,...,2,info@wuzzuf.net,https://wuzzuf.net/terms-and-conditions,Egypt's largest job board. Scraping with rate ...,Egypt,Giza,On-site,"Giza, Egypt",https://wuzzuf.net/jobs/p/ogioemhmsun8-mainten...,2026-01-23
3,7688883585049870245,Civil Engineer,GMC Contracting,Mid,,,,Wuzzuf,wuzzuf,scraping,...,2,info@wuzzuf.net,https://wuzzuf.net/terms-and-conditions,Egypt's largest job board. Scraping with rate ...,Egypt,Cairo,On-site,"Cairo, Egypt",https://wuzzuf.net/jobs/p/3dzott8osopn-civil-e...,2026-01-23
4,-2562285125848060105,Civil Site Engineer,RWAD MISR,Mid,,,,Wuzzuf,wuzzuf,scraping,...,2,info@wuzzuf.net,https://wuzzuf.net/terms-and-conditions,Egypt's largest job board. Scraping with rate ...,Egypt,Alexandria,On-site,"Alexandria, Egypt",https://wuzzuf.net/jobs/p/f2i3dgiygo9q-civil-s...,2026-01-23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1577,2163920522144478897,Staff AI Engineer I,Careem,Mid,,,AI,Greenhouse,greenhouse,official_api,...,10,support@greenhouse.io,https://www.greenhouse.io/terms-of-service,Public job board API. B2B terms apply to emplo...,UAE,,Relocation Required,"Dubai, United Arab Emirates",https://boards.greenhouse.io/careem/jobs/80479...,
1578,-7980064910751830145,Staff Full Stack Engineer,Careem,Mid,,,Full Stack,Greenhouse,greenhouse,official_api,...,10,support@greenhouse.io,https://www.greenhouse.io/terms-of-service,Public job board API. B2B terms apply to emplo...,UAE,,Relocation Required,"Dubai, United Arab Emirates",https://boards.greenhouse.io/careem/jobs/81481...,
1579,2533863659850589771,Staff Software Engineer - Backend (AI Platform),Careem,Mid,,,"Backend, AI",Greenhouse,greenhouse,official_api,...,10,support@greenhouse.io,https://www.greenhouse.io/terms-of-service,Public job board API. B2B terms apply to emplo...,UAE,,Relocation Required,"Dubai, United Arab Emirates",https://boards.greenhouse.io/careem/jobs/81480...,
1580,6360975274312094949,Staff Software Engineer I,Careem,Mid,,,,Greenhouse,greenhouse,official_api,...,10,support@greenhouse.io,https://www.greenhouse.io/terms-of-service,Public job board API. B2B terms apply to emplo...,UAE,,Relocation Required,"Dubai, United Arab Emirates",https://boards.greenhouse.io/careem/jobs/81909...,



✅ The 'df_jobs' DataFrame is ready for use!

   ═══════════════════════════════════════════════════════════════════════════
   COLUMNS REFERENCE:
   ═══════════════════════════════════════════════════════════════════════════
   JOB DATA:       Job_ID, Title, Company, Level, Salary, Experience_Years, Skills
   SOURCE POLICY:  Source, Source_ID, Source_Type, Allowed_Mode, Attribution_Required,
                   Source_URL, Rate_Limit_RPM, Rate_Limit_Burst, Takedown_Contact, 
                   Terms_URL, Source_Notes
   LOCATION:       Country, City, Work_Type, Location
   LINKS:          Apply_URL, Date

   ═══════════════════════════════════════════════════════════════════════════
   QUICK FILTERS:
   ═══════════════════════════════════════════════════════════════════════════
   - Egypt only:         df_jobs[df_jobs['Country'] == 'Egypt']
   - Remote jobs:        df_jobs[df_jobs['Work_Type'] == 'Remote']
   - Cairo jobs:         df_jobs[df_jobs['City'] == 'Cairo']
   - Senior jobs:  

In [18]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              JOBS BY DATE ANALYSIS
# ════════════════════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("📅 JOBS BY DATE - Distribution Analysis")
print("=" * 80)

# Count jobs by date - ORDERED BY COUNT (most to fewest)
jobs_by_date = df_jobs[df_jobs['Date'] != '']['Date'].value_counts()  # Already sorted by count (descending)

print(f"\nTotal jobs with dates: {len(df_jobs[df_jobs['Date'] != ''])}")
print(f"Jobs without dates: {len(df_jobs[df_jobs['Date'] == ''])}")
print("\n")

# Display jobs count per date (ordered by count, most to fewest)
print("📊 Number of Jobs per Date (ordered by count):")
print("-" * 40)
for date, count in jobs_by_date.items():
    print(f"  {date}: {count} jobs")

print("\n")

# Summary statistics
if len(jobs_by_date) > 0:
    dates_sorted = jobs_by_date.index.sort_values()
    print("📈 Summary:")
    print(f"  • Most active day: {jobs_by_date.idxmax()} ({jobs_by_date.max()} jobs)")
    print(f"  • Least active day: {jobs_by_date.idxmin()} ({jobs_by_date.min()} jobs)")
    print(f"  • Date range: {dates_sorted.min()} to {dates_sorted.max()}")
    print(f"  • Average jobs per day: {jobs_by_date.mean():.1f}")

# Create a simple bar chart representation (top 10 dates by count)
print("\n📊 Visual Distribution (Top 10 dates by job count):")
print("-" * 50)
for date, count in jobs_by_date.head(10).items():

    bar = "█" * min(count // 2, 40)  # Scale bars (adjusted for visibility)    print(f"  {date} | {bar} {count}")

📅 JOBS BY DATE - Distribution Analysis

Total jobs with dates: 460
Jobs without dates: 1122


📊 Number of Jobs per Date (ordered by count):
----------------------------------------
  2026-01-23: 38 jobs
  2026-01-22: 20 jobs
  2026-01-20: 12 jobs
  2025-10-23: 12 jobs
  2026-01-19: 11 jobs
  2026-01-16: 8 jobs
  2026-01-14: 6 jobs
  2025-12-17: 6 jobs
  2026-01-21: 6 jobs
  2026-01-18: 6 jobs
  2026-01-17: 5 jobs
  2026-01-13: 5 jobs
  2025-10-20: 5 jobs
  2026-01-11: 4 jobs
  2026-01-06: 4 jobs
  2025-12-23: 4 jobs
  2025-12-08: 4 jobs
  2025-11-19: 4 jobs
  2025-08-15: 4 jobs
  2025-08-12: 4 jobs
  2025-08-06: 4 jobs
  2025-05-20: 4 jobs
  2024-11-12: 4 jobs
  2026-01-15: 3 jobs
  2026-01-05: 3 jobs
  2026-01-04: 3 jobs
  2025-12-18: 3 jobs
  2025-12-14: 3 jobs
  2025-12-11: 3 jobs
  2025-12-04: 3 jobs
  2025-12-02: 3 jobs
  2025-11-25: 3 jobs
  2025-11-18: 3 jobs
  2025-10-16: 3 jobs
  2025-09-21: 3 jobs
  2025-07-30: 3 jobs
  2025-07-24: 3 jobs
  2025-07-07: 3 jobs
  2025-07-03: 3 

In [19]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              HALAL FILTER - Islamic Compliance Filter
# ════════════════════════════════════════════════════════════════════════════════════════
# This filter removes jobs from companies/industries that are not considered Halal
# according to Islamic principles. Industries filtered out include:
#   - Conventional Banking & Interest-based Finance (Riba)
#   - Alcohol & Beverage companies
#   - Gambling & Betting companies
#   - Adult Entertainment
#   - Pork-related businesses
#   - Tobacco companies
#   - Weapons & Arms manufacturers
#   - Conventional Insurance (non-Takaful)
#   - BNPL/Lending Fintech (Egypt Focus)
# ════════════════════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("🕌 HALAL FILTER - Filtering Non-Halal Industries")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════
# NON-HALAL KEYWORDS (Company names, industries, job titles)
# ═══════════════════════════════════════════════════════════════════════════

# Conventional Banks & Interest-based Financial Institutions
HARAM_BANKS = set([
    # International Banks
    "hsbc", "citibank", "citi", "barclays", "jpmorgan", "jp morgan", "chase", 
    "goldman sachs", "morgan stanley", "deutsche bank", "ubs", "credit suisse",
    "bnp paribas", "societe generale", "santander", "ing bank", "rbs", 
    "royal bank of scotland", "lloyds", "natwest", "standard chartered",
    "bank of america", "wells fargo", "capital one", "american express", "amex",
    # Egyptian Conventional Banks
    "national bank of egypt", "nbe", "banque misr", "banque du caire", "cib",
    "commercial international bank", "qnb", "qatar national bank", "alexbank",
    "arab african international bank", "aaib", "credit agricole",
    "attijariwafa", "ahli united bank", "housing development bank", "hdb",
    "export development bank", "edb", "industrial development bank", "idb",
    "egyptian arab land bank", "suez canal bank", "bank audi", "blom bank",
    # Regional Banks
    "emirates nbd", "mashreq", "adcb", "fab", "first abu dhabi", "rakbank",
    "saudi british bank", "sabb", "riyad bank", "arab national bank", "anb",
    "banque saudi fransi", "samba", "ncb", "national commercial bank",
    # Keywords indicating conventional banking
    "interest rate", "loan officer", "mortgage", "credit card", "consumer lending",
    "retail banking", "investment banking", "wealth management",
])

# Alcohol & Beverage Companies
HARAM_ALCOHOL = set([
    "heineken", "budweiser", "anheuser-busch", "ab inbev", "diageo", "pernod ricard",
    "bacardi", "johnnie walker", "smirnoff", "absolut", "jack daniels", "jim beam",
    "corona", "stella artois", "guinness", "carlsberg", "molson coors",
    "constellation brands", "brown forman", "campari",
    "brewery", "breweries", "distillery", "winery", "wine", "beer", "liquor",
    "spirits", "alcoholic", "vodka", "whiskey", "whisky", "rum", "gin", "tequila",
    "champagne", "cognac", "brandy",
])

# Gambling & Betting Companies
HARAM_GAMBLING = set([
    "bet365", "draftkings", "fanduel", "mgm", "caesars", "las vegas sands",
    "wynn", "flutter entertainment", "paddy power", "betfair", "pokerstars",
    "888 holdings", "william hill", "ladbrokes", "betway", "unibet", "bwin",
    "pinnacle", "betsson", "kindred group", "entain", "gvc holdings",
    "scientific games", "igt", "aristocrat", "tabcorp", "sportsbet",
    "casino", "gambling", "betting", "lottery", "poker", "slots", "sportsbook",
    "wagering", "bookmaker", "odds", "blackjack", "roulette",
])

# Adult Entertainment
HARAM_ADULT = set([
    "playboy", "penthouse", "onlyfans", "pornhub", "mindgeek", "aylo",
    "adult entertainment", "adult content", "xxx", "escort",
])

# Pork-related
HARAM_PORK = set([
    "smithfield", "hormel", "tyson pork", "oscar mayer", "jimmy dean",
    "pork", "bacon", "ham", "swine", "pig farm",
])

# Tobacco Companies
HARAM_TOBACCO = set([
    "philip morris", "altria", "british american tobacco", "bat", "imperial brands",
    "japan tobacco", "reynolds american", "lorillard", "marlboro",
    "tobacco", "cigarette", "vaping", "vape", "juul", "e-cigarette",
])

# Weapons & Arms Manufacturers
HARAM_WEAPONS = set([
    "lockheed martin", "raytheon", "northrop grumman", "boeing defense",
    "general dynamics", "bae systems", "l3harris", "leidos",
    "huntington ingalls", "textron", "leonardo drs",
    "weapons", "defense contractor", "arms manufacturer", "missiles",
    "ammunition", "munitions",
])

# Conventional Insurance (Non-Takaful) - Empty by default
HARAM_INSURANCE = set()

# ═══════════════════════════════════════════════════════════════════════════
# HARAM FINTECH & TECH COMPANIES (Egypt Focus)
# ═══════════════════════════════════════════════════════════════════════════

HARAM_FINTECH_EGYPT = set([
    # EGYPTIAN BNPL & LENDING FINTECHS (Interest-based / Riba)
    "valu", "valU",
    "sympl",
    "shahry",
    "khazna",
    "nowpay",
    "kashat",
    "moneyfellows",
    "forsa",
    "contact financial",
    "premium card",
    "b.tech credit",
    "tasaheel",
    "aman microfinance",
    "tanmeyah",
    "sarwa capital",
    "bedaya",
    "tamweely",
    "halan",
    "paymob lending",
    "dopay",
    "lucky",
    "kash2r",
    # REGIONAL BNPL & LENDING FINTECHS
    "tabby",
    "tamara",
    "postpay",
    "spotii",
    "cashew",
    "taly",
    "afterpay",
    "klarna",
    "affirm",
    "clearpay",
    "zip pay",
    "zippay",
    "sezzle",
    "splitit",
    "uplift",
    "bread financial",
    # CRYPTO TRADING/SPECULATION PLATFORMS
    "binance",
    "coinbase",
    "kraken",
    "bitfinex",
    "kucoin",
    "bybit",
    "okx",
    "huobi",
    "ftx",
    "crypto.com",
    "gemini",
    "bitstamp",
    "rain",
    "fasset",
    "bitcoin",
    "cryptocurrency trading",
    "crypto exchange",
    "defi",
    "yield farming",
    # INTEREST-BASED LENDING KEYWORDS
    "payday loan",
    "payday lending",
    "personal loan",
    "personal lending",
    "consumer lending",
    "consumer loan",
    "microlending",
    "microfinance",
    "buy now pay later",
    "bnpl",
    "installment loan",
    "credit scoring",
    "debt collection",
    "collections agency",
    "loan origination",
    "loan servicing",
    "subprime",
    "predatory lending",
    # EGYPTIAN CONVENTIONAL FINANCIAL TECH
    "fawry finance",
    "aman holding",
    "e-finance",
    # DATING APPS & HARAM SOCIAL PLATFORMS
    "tinder",
    "bumble",
    "hinge",
    "badoo",
    "match.com",
    "okcupid",
    "grindr",
    "her app",
    "coffee meets bagel",
    "plenty of fish",
    "pof",
    "dating app",
    "hookup app",
])

# ══════════════════════════════════════════════════════════════════════════
# KNOWN HALAL/ISLAMIC FINTECH (WHITELIST - Do NOT filter these)
# ══════════════════════════════════════════════════════════════════════════
HALAL_FINTECH_WHITELIST = set([
    "bosta",
    "swvl",
    "breadfast",
    "instashop",
    "elmenus",
    "otlob",
    "talabat",
    "vezeeta",
    "paymob",
    "fawry",
    "trella",
    "capiter",
    "cartona",
    "maxab",
    "rabbit",
    "homzmart",
    "brimore",
    "taager",
    "nawy",
    "amenli",
    "almatar",
])

# Combine all non-halal keywords
NON_HALAL_KEYWORDS = (
    HARAM_BANKS | HARAM_ALCOHOL | HARAM_GAMBLING | 
    HARAM_ADULT | HARAM_PORK | HARAM_TOBACCO | HARAM_WEAPONS | 
    HARAM_INSURANCE | HARAM_FINTECH_EGYPT
)

# ═══════════════════════════════════════════════════════════════════════════
# HALAL FILTER FUNCTION
# ═══════════════════════════════════════════════════════════════════════════

import re

def is_halal_job(row):
    """
    Check if a job is from a Halal (permissible) company/industry.
    Returns True if the job appears to be Halal, False if it should be filtered out.
    """
    # Get text to check (company, title, location combined)
    company = str(row.get('Company', '')).lower()
    title = str(row.get('Title', '')).lower()
    location = str(row.get('Location', '')).lower()
    
    # WHITELIST CHECK - Skip filtering for known halal companies
    for halal_company in HALAL_FINTECH_WHITELIST:
        if halal_company.lower() in company:
            return True  # Known halal company, don't filter
    
    # Combine all text for checking
    combined_text = f"{company} {title} {location}"
    
    # Check against non-halal keywords
    for keyword in NON_HALAL_KEYWORDS:
        keyword_lower = keyword.lower()
        if keyword_lower in combined_text:
            # Additional check for short keywords to avoid false positives
            if len(keyword_lower) <= 3:
                if re.search(r'\b' + re.escape(keyword_lower) + r'\b', combined_text):
                    return False
            else:
                return False
    
    return True

# ═══════════════════════════════════════════════════════════════════════════
# APPLY HALAL FILTER
# ═══════════════════════════════════════════════════════════════════════════

print(f"\n📊 Before Halal Filter: {len(df_jobs)} jobs")

# Store original count
original_count = len(df_jobs)

# Create filtered DataFrame
df_jobs_halal = df_jobs[df_jobs.apply(is_halal_job, axis=1)].copy()

# Get removed jobs for reporting
df_removed = df_jobs[~df_jobs.apply(is_halal_job, axis=1)]

filtered_count = len(df_jobs_halal)
removed_count = original_count - filtered_count

print(f"✅ After Halal Filter: {filtered_count} jobs")
print(f"❌ Removed (Non-Halal): {removed_count} jobs")

# ═══════════════════════════════════════════════════════════════════════════
# DISPLAY REMOVED JOBS (for transparency)
# ═══════════════════════════════════════════════════════════════════════════

if removed_count > 0:
    print("\n" + "=" * 80)
    print("🚫 REMOVED JOBS (Non-Halal Industries):")
    print("=" * 80)
    
    # Group removed jobs by reason/company
    removed_companies = df_removed['Company'].value_counts()
    print(f"\nRemoved by Company ({len(removed_companies)} companies):")
    for company, count in removed_companies.items():
        print(f"   ❌ {company}: {count} job(s)")
    
    print("\n" + "-" * 40)
    print("Removed Jobs Details:")
    print(df_removed[['Title', 'Company', 'Country', 'Source']].to_string())

# ═══════════════════════════════════════════════════════════════════════════
# UPDATE MAIN DATAFRAME
# ═══════════════════════════════════════════════════════════════════════════

# Replace df_jobs with the halal-filtered version
df_jobs = df_jobs_halal

print("\n" + "=" * 80)
print("✅ HALAL FILTER APPLIED SUCCESSFULLY")
print("=" * 80)
print(f"""
   🕌 The 'df_jobs' DataFrame now contains only Halal-compliant jobs.
   
   📊 Summary:
      • Original jobs: {original_count}
      • Halal jobs: {filtered_count}
      • Removed: {removed_count}
   
   🔄 To get the original unfiltered data, re-run the aggregator cells.
   
   📋 Industries Filtered Out:
      • Conventional Banks (Interest/Riba-based)
      • Alcohol & Beverage companies
      • Gambling & Betting companies
      • Adult Entertainment
      • Pork-related businesses
      • Tobacco companies
      • Weapons manufacturers
      • BNPL/Lending Fintech (valU, Sympl, Khazna, Shahry, etc.)
      • Crypto Trading Platforms (Binance, Coinbase, etc.)
      • Interest-based Microfinance
      • Dating Apps (Tinder, Bumble, etc.)
   
   🇪🇬 Egyptian Fintech Companies Filtered:
      • valU (BNPL with interest)
      • Sympl (BNPL platform)
      • Shahry (BNPL for e-commerce)
      • Khazna (Consumer lending)
      • NowPay (Salary advance with fees)
      • Kashat (Microlending)
      • Halan (Lending products)
      • Contact Financial (Consumer lending)
      • Tasaheel, Tanmeyah, Aman (Microfinance)
   
   ✅ Halal Egyptian Companies NOT filtered:
      • Bosta, Swvl, Breadfast, Vezeeta, Paymob, Fawry
      • Trella, Capiter, Cartona, Rabbit, Homzmart
      • Brimore, Taager, Nawy, Elmenus, Talabat
   
   ℹ️ Note: Review the removed jobs list and adjust the filter sets
      (HARAM_FINTECH_EGYPT or HALAL_FINTECH_WHITELIST) as needed.
""")

🕌 HALAL FILTER - Filtering Non-Halal Industries

📊 Before Halal Filter: 1582 jobs
✅ After Halal Filter: 1553 jobs
❌ Removed (Non-Halal): 29 jobs

🚫 REMOVED JOBS (Non-Halal Industries):

Removed by Company (20 companies):
   ❌ Dopay 8: 7 job(s)
   ❌ Affirm: 2 job(s)
   ❌ General Dynamics Information Technology: 2 job(s)
   ❌ Mindrift: 2 job(s)
   ❌ Tabby: 1 job(s)
   ❌ Mashreq Corporate & Investment Banking Group: 1 job(s)
   ❌ CEQUENS: 1 job(s)
   ❌ EDC Consulting: 1 job(s)
   ❌ CEDENT: 1 job(s)
   ❌ LARC Staffing, LLC: 1 job(s)
   ❌ GTN Technical Staffing: 1 job(s)
   ❌ Mastech Digital: 1 job(s)
   ❌ Capital One: 1 job(s)
   ❌ Allwyn Lottery Solutions: 1 job(s)
   ❌ ChaseSource: 1 job(s)
   ❌ Datadog: 1 job(s)
   ❌ Magic Media: 1 job(s)
   ❌ Kraken: 1 job(s)
   ❌ Match Group: 1 job(s)
   ❌ Binance: 1 job(s)

----------------------------------------
Removed Jobs Details:
                                                                                     Title                          

In [20]:
# ════════════════════════════════════════════════════════════════════════════════════════
#                              EXPORT DATASET (CSV/XLSX)
# ════════════════════════════════════════════════════════════════════════════════════════
# Saves the final df_jobs DataFrame to CSV and XLSX in the workspace folder.

from importlib.util import find_spec
import sys, subprocess

# Ensure openpyxl for Excel export
if find_spec("openpyxl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"]) 

csv_path = "Egypt_Tech_Jobs.csv"
xlsx_path = "Egypt_Tech_Jobs.xlsx"

# Save to CSV (UTF-8 BOM for Excel compatibility)
df_jobs.to_csv(csv_path, index=False, encoding="utf-8-sig")

# Save to XLSX
try:
    df_jobs.to_excel(xlsx_path, index=False)
    excel_msg = f"✅ XLSX saved: {xlsx_path}"
except Exception as e:
    excel_msg = f"⚠️ XLSX export failed: {e}"

print("=" * 80)
print("💾 EXPORT COMPLETE")
print("=" * 80)
print(f"CSV: {csv_path}")
print(excel_msg)


💾 EXPORT COMPLETE
CSV: Egypt_Tech_Jobs.csv
✅ XLSX saved: Egypt_Tech_Jobs.xlsx
